# XAI stability and faithfulness metrics -- RIS, ROS, RES, PGI, and LAP

In [1]:
# Defs for 
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        LRP
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion
#        LIME
#        KernelSHAP
#        GradientSHAP

In [2]:
import sys

!{sys.executable} -m pip install nbimporter

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
#!{sys.executable} -m pip install -e OpenXAI

In [3]:
import numpy as np
import pandas as pd
import nbimporter
import time
import warnings

# Utils
import torch
import torch.nn as nn
import os
from sklearn.utils import shuffle

import xai_explainers_for_Captum as explainer

import my_progressbar as pb

# NormalPerturbation

In [4]:
#import openxai

# Perturbation methods required for the computation of the relative stability metrics
#from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
#from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation

In [5]:
# class from OpenXAI (Agarwal, Chirag, et al., 2022)
class BasePerturbation:
    '''
    Base Class for perturbation methods.
    '''
    
    def __init__(self, data_format):
        '''
        Initialize generic parameters for the perturbation method
        '''
        self.data_format = data_format
    
    def get_perturbed_inputs(self):
        '''
        This function implements the logic of the perturbation methods which will return perturbed samples.
        '''
        pass
    
class NormalPerturbation(BasePerturbation):
    def __init__(self, data_format, mean: int = 0, std_dev: float = 0.05, flip_percentage: float = 0.3):
        self.mean = mean
        self.std_dev = std_dev
        self.flip_percentage = flip_percentage

        super(NormalPerturbation, self).__init__(data_format)
        '''
        Initializes the marginal perturbation method where each column is sampled from marginal distributions 
        given per variable. dist_per_feature : vector of distribution generators 
        (tdist under torch.distributions).
        Note : These distributions are assumed to have zero mean since they get added to the original sample.
        '''
        pass

    def get_perturbed_inputs(self, original_sample: torch.FloatTensor, feature_mask: torch.BoolTensor,
                             num_samples: int, feature_metadata: list, max_distance: int = None) -> torch.tensor:
        '''
        feature mask : this indicates the static features
        num_samples : number of perturbed samples.
        max_distance : the maximum distance between original sample and purturbed samples.
        '''
        feature_type = feature_metadata
        assert len(feature_mask) == len(
            original_sample), "mask size == original sample in get_perturbed_inputs for {}".format(self.__class__)

        perturbed_cols = []
        continuous_features = torch.tensor([i == 'c' for i in feature_type])
        discrete_features = torch.tensor([i == 'd' for i in feature_type])

        # Processing continuous columns
        mean = self.mean
        std_dev = self.std_dev
        perturbations = torch.normal(mean, std_dev,
                                     [num_samples, len(feature_type)]) * continuous_features + original_sample

        # Processing discrete columns
        flip_percentage = self.flip_percentage
        p = torch.empty(num_samples, len(feature_type)).fill_(flip_percentage)
        perturbations = perturbations * (~discrete_features) + torch.abs(
            (perturbations * discrete_features) - (torch.bernoulli(p) * discrete_features))

        # keeping features static that are in top-K based on feature mask
        perturbed_samples = original_sample * feature_mask + perturbations * (~feature_mask)

        return perturbed_samples

# Quantitative Metrics -- Auxiliar Methods

In [6]:
# convert log odds values to predicted classes in a binary classification problem

def log_odds_to_binary_class(lodds):
    
    lodds= lodds.detach().numpy()
    
    p= np.exp(lodds) / (1 + np.exp(lodds))
    
    classes= np.zeros(p.shape)
    
    for i, prob in enumerate(p):    
        if (prob>= 0.5): classes[i]= 1
    
    return torch.tensor(classes)

In [7]:
# RETURN tensor_x and tensor_y datsets (tensors) according to euclidean distance ordering from target_x

def distance_ordering(tensor_x, tensor_y, target_x):
    
    # Calculate Euclidean distances for each row
    distances= torch.norm((tensor_x - target_x), dim=1)

    # Sort the data tensor based on distances
    sorted_indices= torch.argsort(distances)
    
    sorted_x= tensor_x[sorted_indices]
    sorted_y= tensor_y[sorted_indices]
    
    return sorted_x, sorted_y

In [8]:
# Remove a set of rows in a tensor dataset by index

# dataset is a n elements dataset
# index_to_remove is a m elements tensor with the indexes to remove

# RETURN a subset form dataset without the index_to_remove instances
    
def remove_tensor_row_by_indexset(dataset, index_to_remove):

    n_rows= index_to_remove.shape[0]
    
    index_to_remove= index_to_remove.sort().values
    
    subset= dataset.clone()
    
    for i in range(n_rows):
        row_exclude= index_to_remove[i]-i
    
        subset= torch.cat((subset[:row_exclude],subset[row_exclude+1:]))

    return subset

In [9]:
# Get a subset from a dataset with at least n_elements

# x is a tensor instance
# x_class is a tensor with the class of x
# dataset is a m elements tensor dataset
# dataset_class is a m elements tensor with the predicted 
# n_elements is an integer indicating the size of the subset

# RETURN two tensor subsets (from dataset and dataset_class) with 
#        option 1 - n_elements ordered first by class (same from x) and then by distance from x
#        option 0 - n_elements ordered by class (same from x) and filled (if necessary) with x and x_class

# option 0 gives us y' = y for all x' and option 1 relaxes such a restriction

def get_subsets(x, x_class, dataset, dataset_class, n_elements, option:int=0):
    data_size= dataset.shape[0]
    
    if (data_size< n_elements):
        raise ValueError("Data size must be greater than n_elements!")
    else:
        if (option):
            # order the dataset and dataset_class by distance from x
            dataset_order, dataset_class_order= distance_ordering(dataset, dataset_class, x.unsqueeze(0))
        
            # get the subset with first num_perts points by the same x class
            ind_same_class= (x_class == dataset_class_order).nonzero()[:n_elements].squeeze()
        
            subset= torch.index_select(input=dataset_order, dim=0, index=ind_same_class)
            subset_class= torch.index_select(input=dataset_class_order, dim=0, index=ind_same_class)
        else:
            # get the subset with first num_perts points by the same x class
            ind_same_class= (x_class == dataset_class).nonzero()[:n_elements].squeeze()
            
            # get only the elements in dataset under y' = y
            subset= torch.index_select(input=dataset, dim=0, index=ind_same_class)
            subset_class= torch.index_select(input=dataset_class, dim=0, index=ind_same_class)
            
            
        # if there are no elements enough in dataset matching with x_class
        if (subset.shape[0]< n_elements):
            last= n_elements - subset.shape[0]
            
            # we complete the n_elements of subset with the first instances of the ordered dataset
            if (option):
                if (ind_same_class.numel()== 1): # avoid a breaking when only one element matches with x_class
                    ind_same_class= torch.tensor([ind_same_class])

                dataset_order= remove_tensor_row_by_indexset(dataset_order, ind_same_class)
                dataset_class_order= remove_tensor_row_by_indexset(dataset_class_order, ind_same_class)

                dataset_order= dataset_order[0:last,:]
                dataset_class_order= dataset_class_order[0:last]

                subset= torch.cat((subset, dataset_order))
                subset_class= torch.cat((subset_class.reshape(-1), dataset_class_order.reshape(-1)))
            
            # or we complete with x and x_class
            else:
                fill_x, fill_x_class= [], []
                
                for i in range(last):
                    fill_x.append(x)
                    fill_x_class.append(x_class)
                
                fill_x= torch.stack(fill_x)
                fill_x_class= torch.stack(fill_x_class)
                
                subset= torch.cat((subset, fill_x))
                subset_class= torch.cat((subset_class.reshape(-1), fill_x_class.reshape(-1)))
            
            
        return subset, subset_class

In [70]:
ds_x= torch.tensor([[0.5885, 0.5961, 0.8704, 0.3738],
                    [0.5884, 0.5914, 0.8719, 0.3758],
                    [0.5905, 0.5933, 0.8656, 0.3686],
                    [0.5978, 0.5950, 0.8616, 0.3753],
                    [0.5881, 0.5873, 0.8663, 0.3726],
                    [0.5929, 0.5886, 0.8684, 0.3712],
                    [0.5914, 0.5882, 0.8652, 0.3822],
                    [0.5849, 0.6006, 0.8690, 0.3694],
                    [0.5849, 0.6032, 0.8677, 0.3707],
                    [0.5855, 0.5927, 0.8652, 0.3665],
                    [0.5841, 0.5866, 0.8631, 0.3747],
                    [0.5826, 0.6017, 0.8672, 0.3829],
                    [0.5848, 0.5873, 0.8670, 0.3827],
                    [0.5868, 0.6030, 0.8727, 0.3816],
                    [0.5804, 0.5916, 0.8600, 0.3738],
                    [0.5806, 0.6007, 0.8622, 0.3812],
                    [0.5885, 0.5875, 0.8711, 0.3688],
                    [0.5843, 0.5850, 0.8636, 0.3737],
                    [0.5833, 0.6043, 0.8595, 0.3764],
                    [0.5859, 0.6031, 0.8745, 0.3710],
                    [0.5995, 0.5965, 0.8580, 0.3724],
                    [0.5843, 0.5905, 0.8743, 0.3685],
                    [0.5829, 0.5999, 0.8645, 0.3653],
                    [0.5975, 0.5856, 0.8650, 0.3712],
                    [0.5912, 0.5915, 0.8624, 0.3636],
                    [0.5977, 0.6029, 0.8673, 0.3677],
                    [0.6002, 0.5916, 0.8576, 0.3744],
                    [0.5869, 0.5977, 0.8715, 0.3891],
                    [0.5946, 0.5898, 0.8552, 0.3822],
                    [0.6014, 0.5905, 0.8728, 0.3767]])

ds_y= torch.tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
                    0, 0, 0, 0, 0, 0, 0, 1, 0, 0])

x_ponto= torch.tensor([0.5897, 0.5957, 0.8659, 0.3762])
x_ponto_class= torch.tensor([1])

get_subsets(x_ponto, x_ponto_class, ds_x, ds_y, 10)

(tensor([[0.5885, 0.5961, 0.8704, 0.3738],
         [0.5869, 0.5977, 0.8715, 0.3891],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762]]),
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [71]:
get_subsets(x_ponto, x_ponto_class, ds_x, ds_y, 10, 1)

(tensor([[0.5885, 0.5961, 0.8704, 0.3738],
         [0.5869, 0.5977, 0.8715, 0.3891],
         [0.5884, 0.5914, 0.8719, 0.3758],
         [0.5905, 0.5933, 0.8656, 0.3686],
         [0.5978, 0.5950, 0.8616, 0.3753],
         [0.5881, 0.5873, 0.8663, 0.3726],
         [0.5929, 0.5886, 0.8684, 0.3712],
         [0.5914, 0.5882, 0.8652, 0.3822],
         [0.5849, 0.6006, 0.8690, 0.3694],
         [0.5849, 0.6032, 0.8677, 0.3707]]),
 tensor([1, 1, 0, 0, 0, 0, 0, 0, 0, 0]))

In [10]:
# clip values near to zero in v replacing by eps

# v is a single value (float) or a numpy.ndarray with (n,) shape
# eps is a small number of tolerance limiting what is a small value

# RETURN v clipped

def clip_small_values(v, eps=1e-6):
    
    v_aux= v.copy()
    
    if (type(v_aux)== np.ndarray):
        elements= v_aux.shape[0]

        for i in range(elements):

            if (v_aux[i]< 0 and np.abs(v_aux[i])< eps):
                v_aux[i]= -eps
            elif (v_aux[i]> 0 and v_aux[i]< eps):
                v_aux[i]= eps
    else:
        if (v_aux< 0 and np.abs(v_aux)< eps):
            v_aux[i]= -eps
        elif (v_aux> 0 and v_aux< eps):
            v_aux= eps
                        
    return v_aux

In [11]:
# returns the square of the difference of any two quantities v1 and v2.

def square_difference(v1, v2):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    return np.power(dif_flat, 2)

In [12]:
# returns the Lp norm of the difference between v1 and v2.
# normalizes the difference between v1 and v2 by v1 (adapted; Agarwal, Chirag, et al., 2022)

def lp_norm_dif(v1, v2, p_norm=2, eps=1e-6, norm:bool=True):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    #if (norm==True): print('dif before div', dif_flat)
    
    if (norm==True):
        #v1_flat= np.clip(v1_flat, eps, None)
        v1_flat= clip_small_values(v1_flat, eps)
        
        dif_flat= np.divide(dif_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)
        
        #v2_flat= clip_small_values(v2_flat, eps)
        #dif_flat= 1 - np.divide(v2_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)
        
    #if (norm==True): print('dif afterr div', dif_flat)

    return np.linalg.norm(dif_flat, ord=p_norm)

In [13]:
# compute norm between predictions per perturbation - RIS

def ris_measure(x_data, x_pert, exp_data, exp_pert, p_norm=2, eps=1e-6):
    
    x_dif_norm= lp_norm_dif(x_data, x_pert, p_norm=p_norm, eps=eps, norm=True)
    #x_dif_norm= np.clip(x_dif_norm, eps, None)
    x_dif_norm= clip_small_values(x_dif_norm, eps)
    
    exp_dif_norm= lp_norm_dif(exp_data, exp_pert, p_norm=p_norm, eps=eps, norm=True)
    
    stability_measure= np.divide(exp_dif_norm, x_dif_norm, where=x_dif_norm!=0)
    
    return stability_measure

In [14]:
# compute norm between representations - ROS

def ros_measure(fx_data, fx_pert, exp_data, exp_pert, p_norm=2, eps=1e-6):
    
    fx_data= fx_data.detach()
    fx_pert= fx_pert.detach()
    
    fx_dif_norm= lp_norm_dif(fx_data, fx_pert, p_norm=p_norm, eps=eps, norm=True)
    #fx_dif_norm= np.clip(fx_dif_norm, eps, None)
    fx_dif_norm= clip_small_values(fx_dif_norm, eps)
    
    exp_dif_norm= lp_norm_dif(exp_data, exp_pert, p_norm=p_norm, eps=eps, norm=True)

    stability_measure= np.divide(exp_dif_norm, fx_dif_norm, where=fx_dif_norm!=0)
    
    return stability_measure

In [15]:
# model is a trained PyTorch classifier
# data and label refer to the data instance under explanation and its label as tensor lines
# descriptor defines the explainers used and their parameters

# RETURN explanations for one data instance (target_x) from explainers
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        LRP
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion
#        LIME
#        KernelSHAP
#        GradientSHAP

def get_x_explanations(model, data, label, descriptor):
    
    itGd_x_exp= torch.zeros(data.size()).reshape(-1)
    iXGd_x_exp= torch.zeros(data.size()).reshape(-1)
    dLif_x_exp= torch.zeros(data.size()).reshape(-1)
    lwrp_x_exp= torch.zeros(data.size()).reshape(-1)
    smoo_x_exp= torch.zeros(data.size()).reshape(-1)
    vnGd_x_exp= torch.zeros(data.size()).reshape(-1)
    gdBp_x_exp= torch.zeros(data.size()).reshape(-1)
    occl_x_exp= torch.zeros(data.size()).reshape(-1)
    lime_x_exp= torch.zeros(data.size()).reshape(-1)
    kshp_x_exp= torch.zeros(data.size()).reshape(-1)
    gshp_x_exp= torch.zeros(data.size()).reshape(-1)
    
    exps= descriptor['methods']
    
    # ------------------------------------ x explanation
    # ignore non-important warnings temporarily
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Setting")

        x_exp= explainer.Explainers()
        target_x= torch.clone(data)
        if ('IntegratedGradients' in exps): itGd_x_exp= x_exp.int_grad(model, target_x)
        
        target_x= torch.clone(data)
        if ('InputXGradient' in exps): iXGd_x_exp= x_exp.inx_grad(model, target_x)
        
        target_x= torch.clone(data)
        if ('DeepLift' in exps): dLif_x_exp= x_exp.dp_lift(model, target_x)
        
        target_x= torch.clone(data)
        if ('LRP' in exps): lwrp_x_exp= x_exp.lrp(model, target_x)
        
        target_x= torch.clone(data)                                            # IntGrad is used here
        if ('SmoothGrad' in exps): smoo_x_exp= x_exp.smo_grad(model, target_x, method=False)
        
        target_x= torch.clone(data)
        if ('VanillaGrad' in exps): vnGd_x_exp= x_exp.vnl_grad(model, target_x)
        
        target_x= torch.clone(data)
        if ('GuidedBackprop' in exps): gdBp_x_exp= x_exp.g_bkprop(model, target_x)
        
        target_x= torch.clone(data)
        if ('Occlusion' in exps): occl_x_exp= x_exp.occ(model, target_x)
        
        target_x= torch.clone(data)
        if ('Lime' in exps): lime_x_exp= x_exp.lime(model, target_x)
        
        target_x= torch.clone(data)
        if ('KernelShap' in exps): kshp_x_exp= x_exp.k_shap(model, target_x)
        
        target_x= torch.clone(data)
        if ('GradientShap' in exps): gshp_x_exp= x_exp.g_shap(model, target_x)

    # reset the warning settings
    warnings.resetwarnings()
        
    importances= {
        'itGd': itGd_x_exp, 'iXGd': iXGd_x_exp, 'dLif': dLif_x_exp, 'lwrp': lwrp_x_exp, 
        'smoo': smoo_x_exp, 'vnGd': vnGd_x_exp, 'gdBp': gdBp_x_exp, 'occl': occl_x_exp,
        'lime': lime_x_exp, 'k_shap': kshp_x_exp, 'g_shap': gshp_x_exp,  
    }
        
    return importances

# Metric -- Relative Input/Output Stability -- RIS / ROS

In [16]:
# Relative Input/Output Stability
# model is a trained PyTorch classifier
# data is a preprocessed Pandas DataFrame to evaluate
# labels are the respective data labels (Pandas DataFrame)
# perturbation is a OpenXAI perturbation object
# descriptor defines the parameters to explanations and data perturbations
# stab_type can be: 'ris' to calculate only RIS metric, 'ros' to ROS metric, and 'both' for RIS and ROS

# approximates the maximum L-p distance between explanations in a neighborhood around input x
# RETURN RIS max/mean and ROS max/mean values for 
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        LRP
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion
#        LIME
#        KernelSHAP
#        GradientSHAP

# The larger the RIS/ROS values of the underlying explanation method, the more unstable the method 
# to input/output perturbations.

# see Agarwal, Chirag, et al. "OpenXAI: Towards a transparent evaluation of model explanations,"
# Alvarez Melis, David, and Tommi Jaakkola. "Towards robust interpretability with self-explaining 
# neural networks," and Ortigossa, Evandro S., Thales Gonçalves, and Luis Gustavo Nonato. "EXplainable 
# Artificial Intelligence (XAI)–From Theory to Methods and Applications."

def relative_stability(model, data, labels, perturbation, descriptor, stab_type='both'):
    
    itGd_ris_max_ratios= []
    iXGd_ris_max_ratios= []
    dLif_ris_max_ratios= []
    lwrp_ris_max_ratios= []
    smoo_ris_max_ratios= []
    vnGd_ris_max_ratios= []
    gdBp_ris_max_ratios= []
    occl_ris_max_ratios= []
    lime_ris_max_ratios= []
    sknl_ris_max_ratios= []
    sgrd_ris_max_ratios= []

    itGd_ris_mean_ratios= []
    iXGd_ris_mean_ratios= []
    dLif_ris_mean_ratios= []
    lwrp_ris_mean_ratios= []
    smoo_ris_mean_ratios= []
    vnGd_ris_mean_ratios= []
    gdBp_ris_mean_ratios= []
    occl_ris_mean_ratios= []
    lime_ris_mean_ratios= []
    sknl_ris_mean_ratios= []
    sgrd_ris_mean_ratios= []

    itGd_ros_max_ratios= []
    iXGd_ros_max_ratios= []
    dLif_ros_max_ratios= []
    lwrp_ros_max_ratios= []
    smoo_ros_max_ratios= []
    vnGd_ros_max_ratios= []
    gdBp_ros_max_ratios= []
    occl_ros_max_ratios= []
    lime_ros_max_ratios= []
    sknl_ros_max_ratios= []
    sgrd_ros_max_ratios= []

    itGd_ros_mean_ratios= []
    iXGd_ros_mean_ratios= []
    dLif_ros_mean_ratios= []
    lwrp_ros_mean_ratios= []
    smoo_ros_mean_ratios= []
    vnGd_ros_mean_ratios= []
    gdBp_ros_mean_ratios= []
    occl_ros_mean_ratios= []
    lime_ros_mean_ratios= []
    sknl_ros_mean_ratios= []
    sgrd_ros_mean_ratios= []
    
        
    data_size= data.shape[0]
    
    for i_data in pb.progressbar(range(data_size), "Progress: ", 40):
        
        # i_data and its label as pd.DataFrames
        target_x= pd.DataFrame(data=[data.iloc[i_data,:]], columns=data.columns)
        target_y= pd.DataFrame(data=[labels.iloc[i_data]], columns=labels.columns)
        
        # i_data and its label as tensors
        x_data= torch.tensor(np.asarray(target_x), dtype=torch.float64)
        y_data= torch.tensor(np.asarray(target_y), dtype=torch.float64)
        
        
        # data point prediction
        fx_data= model(x_data)
        y_pred = log_odds_to_binary_class(fx_data)
        
        
        # ------------------------------------ get x_data explanation
        x_exps= get_x_explanations(model, x_data, y_data, descriptor)
        
        itGd_x_exp= x_exps['itGd']
        iXGd_x_exp= x_exps['iXGd']
        dLif_x_exp= x_exps['dLif']
        lwrp_x_exp= x_exps['lwrp']
        smoo_x_exp= x_exps['smoo']
        vnGd_x_exp= x_exps['vnGd']
        gdBp_x_exp= x_exps['gdBp']
        occl_x_exp= x_exps['occl']
        lime_x_exp= x_exps['lime']
        sknl_x_exp= x_exps['k_shap']
        sgrd_x_exp= x_exps['g_shap']
        
       
        # ------------------------------------ x_data perturbation
        # data point perturbation
        mask= torch.zeros(x_data.reshape(-1).shape, dtype=torch.bool)
        
        x_pert_samples= perturbation.get_perturbed_inputs(original_sample=x_data.reshape(-1),
                                                          feature_mask=mask,
                                                          num_samples=descriptor['num_samples'],
                                                          max_distance=descriptor['pert_max_distance'],
                                                          feature_metadata=descriptor['feature_metadata'])
        
        # --- take the closest num_perts points to x_data that have the same predicted class label to x_data
        fx_pert= model(x_pert_samples)
        y_pert_preds= log_odds_to_binary_class(fx_pert)
        
        # get only the first num_perts points ordered by class and distance from x_data
        x_pert_samples, y_pert_preds= get_subsets(x_data.reshape(-1), y_pred.reshape(-1), 
                                                  x_pert_samples, y_pert_preds.reshape(-1), 
                                                  descriptor['num_perts'])
        
        
        # ------------------------------------ explain each x_data perturbation
        itGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
        iXGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
        dLif_exp_pert_samples= torch.zeros_like(x_pert_samples)
        lwrp_exp_pert_samples= torch.zeros_like(x_pert_samples)
        smoo_exp_pert_samples= torch.zeros_like(x_pert_samples)
        vnGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
        gdBp_exp_pert_samples= torch.zeros_like(x_pert_samples)
        occl_exp_pert_samples= torch.zeros_like(x_pert_samples)
        sknl_exp_pert_samples= torch.zeros_like(x_pert_samples)
        sgrd_exp_pert_samples= torch.zeros_like(x_pert_samples)
        lime_exp_pert_samples= torch.zeros_like(x_pert_samples)
                
    
        itGd_x_ris_ratios= []
        iXGd_x_ris_ratios= []
        dLif_x_ris_ratios= []
        lwrp_x_ris_ratios= []
        smoo_x_ris_ratios= []
        vnGd_x_ris_ratios= []
        gdBp_x_ris_ratios= []
        occl_x_ris_ratios= []
        sknl_x_ris_ratios= []
        sgrd_x_ris_ratios= []
        lime_x_ris_ratios= []

        itGd_x_ros_ratios= []
        iXGd_x_ros_ratios= []
        dLif_x_ros_ratios= []
        lwrp_x_ros_ratios= []
        smoo_x_ros_ratios= []
        vnGd_x_ros_ratios= []
        gdBp_x_ros_ratios= []
        occl_x_ros_ratios= []
        sknl_x_ros_ratios= []
        sgrd_x_ros_ratios= []
        lime_x_ros_ratios= []
        
        
        # For each perturbation, calculate the explanation
        for i, x_pert in enumerate(x_pert_samples):
            
            df_x_pert= pd.DataFrame(data=[x_pert.numpy()], columns=data.columns)
            df_y_pert= pd.DataFrame(data=[np.int64(y_pert_preds[i])], columns=labels.columns)
            
            
            # ------------------------------------ get x_pert explanation
            x_exp_pert= get_x_explanations(model, torch.reshape(x_pert, x_data.size()), y_pert_preds[i], 
                                           descriptor)
            
            itGd_exp_pert_samples[i, :]= x_exp_pert['itGd']
            iXGd_exp_pert_samples[i, :]= x_exp_pert['iXGd']
            dLif_exp_pert_samples[i, :]= x_exp_pert['dLif']
            lwrp_exp_pert_samples[i, :]= x_exp_pert['lwrp']
            smoo_exp_pert_samples[i, :]= x_exp_pert['smoo']
            vnGd_exp_pert_samples[i, :]= x_exp_pert['vnGd']
            gdBp_exp_pert_samples[i, :]= x_exp_pert['gdBp']
            occl_exp_pert_samples[i, :]= x_exp_pert['occl']
            sknl_exp_pert_samples[i, :]= x_exp_pert['k_shap']
            sgrd_exp_pert_samples[i, :]= x_exp_pert['g_shap']
            lime_exp_pert_samples[i, :]= x_exp_pert['lime']
            
            
            # ------------------------------------ get stability for each explanator and x_data perturbation
            # RIS
            if not ('ros' in stab_type):
                itGd_ris_measure= ris_measure(x_data, x_pert, 
                                             itGd_x_exp, itGd_exp_pert_samples[i], 
                                             p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                iXGd_ris_measure= ris_measure(x_data, x_pert, 
                                              iXGd_x_exp, iXGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                dLif_ris_measure= ris_measure(x_data, x_pert, 
                                              dLif_x_exp, dLif_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                lwrp_ris_measure= ris_measure(x_data, x_pert, 
                                              lwrp_x_exp, lwrp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                smoo_ris_measure= ris_measure(x_data, x_pert, 
                                              smoo_x_exp, smoo_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                vnGd_ris_measure= ris_measure(x_data, x_pert, 
                                              vnGd_x_exp, vnGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                gdBp_ris_measure= ris_measure(x_data, x_pert, 
                                              gdBp_x_exp, gdBp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                occl_ris_measure= ris_measure(x_data, x_pert, 
                                              occl_x_exp, occl_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                sknl_ris_measure= ris_measure(x_data, x_pert, 
                                              sknl_x_exp, sknl_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                sgrd_ris_measure= ris_measure(x_data, x_pert, 
                                              sgrd_x_exp, sgrd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                lime_ris_measure= ris_measure(x_data, x_pert, 
                                              lime_x_exp, lime_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            
            # ROS
            if not ('ris' in stab_type):
                itGd_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              itGd_x_exp, itGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                iXGd_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              iXGd_x_exp, iXGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                dLif_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              dLif_x_exp, dLif_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                lwrp_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              lwrp_x_exp, lwrp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                smoo_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              smoo_x_exp, smoo_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                vnGd_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              vnGd_x_exp, vnGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                gdBp_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              gdBp_x_exp, gdBp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                occl_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              occl_x_exp, occl_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                sknl_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              sknl_x_exp, sknl_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                sgrd_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              sgrd_x_exp, sgrd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])

                lime_ros_measure= ros_measure(fx_data, fx_pert[i], 
                                              lime_x_exp, lime_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            
            # --- stability measures for each x_data perturbation --- one processing cicle
            # RIS
            if not ('ros' in stab_type):
                itGd_x_ris_ratios.append(itGd_ris_measure)
                iXGd_x_ris_ratios.append(iXGd_ris_measure)
                dLif_x_ris_ratios.append(dLif_ris_measure)
                lwrp_x_ris_ratios.append(lwrp_ris_measure)
                smoo_x_ris_ratios.append(smoo_ris_measure)
                vnGd_x_ris_ratios.append(vnGd_ris_measure)
                gdBp_x_ris_ratios.append(gdBp_ris_measure)
                occl_x_ris_ratios.append(occl_ris_measure)
                sknl_x_ris_ratios.append(sknl_ris_measure)
                sgrd_x_ris_ratios.append(sgrd_ris_measure)
                lime_x_ris_ratios.append(lime_ris_measure)
            
            # ROS
            if not ('ris' in stab_type):
                itGd_x_ros_ratios.append(itGd_ros_measure)
                iXGd_x_ros_ratios.append(iXGd_ros_measure)
                dLif_x_ros_ratios.append(dLif_ros_measure)
                lwrp_x_ros_ratios.append(lwrp_ros_measure)
                smoo_x_ros_ratios.append(smoo_ros_measure)
                vnGd_x_ros_ratios.append(vnGd_ros_measure)
                gdBp_x_ros_ratios.append(gdBp_ros_measure)
                occl_x_ros_ratios.append(occl_ros_measure)
                sknl_x_ros_ratios.append(sknl_ros_measure)
                sgrd_x_ros_ratios.append(sgrd_ros_measure)
                lime_x_ros_ratios.append(lime_ros_measure)
                
        
        # --- append only the max/mean value related to each x_data processed
        # max values
        if not ('ros' in stab_type):
            itGd_ris_max_ratios.append(itGd_x_ris_ratios[np.argmax(itGd_x_ris_ratios)])
            iXGd_ris_max_ratios.append(iXGd_x_ris_ratios[np.argmax(iXGd_x_ris_ratios)])
            dLif_ris_max_ratios.append(dLif_x_ris_ratios[np.argmax(dLif_x_ris_ratios)])
            lwrp_ris_max_ratios.append(lwrp_x_ris_ratios[np.argmax(lwrp_x_ris_ratios)])
            smoo_ris_max_ratios.append(smoo_x_ris_ratios[np.argmax(smoo_x_ris_ratios)])
            vnGd_ris_max_ratios.append(vnGd_x_ris_ratios[np.argmax(vnGd_x_ris_ratios)])
            gdBp_ris_max_ratios.append(gdBp_x_ris_ratios[np.argmax(gdBp_x_ris_ratios)])
            occl_ris_max_ratios.append(occl_x_ris_ratios[np.argmax(occl_x_ris_ratios)])
            sknl_ris_max_ratios.append(sknl_x_ris_ratios[np.argmax(sknl_x_ris_ratios)])
            sgrd_ris_max_ratios.append(sgrd_x_ris_ratios[np.argmax(sgrd_x_ris_ratios)])
            lime_ris_max_ratios.append(lime_x_ris_ratios[np.argmax(lime_x_ris_ratios)])
        
        if not ('ris' in stab_type):
            itGd_ros_max_ratios.append(itGd_x_ros_ratios[np.argmax(itGd_x_ros_ratios)])
            iXGd_ros_max_ratios.append(iXGd_x_ros_ratios[np.argmax(iXGd_x_ros_ratios)])
            dLif_ros_max_ratios.append(dLif_x_ros_ratios[np.argmax(dLif_x_ros_ratios)])
            lwrp_ros_max_ratios.append(lwrp_x_ros_ratios[np.argmax(lwrp_x_ros_ratios)])
            smoo_ros_max_ratios.append(smoo_x_ros_ratios[np.argmax(smoo_x_ros_ratios)])
            vnGd_ros_max_ratios.append(vnGd_x_ros_ratios[np.argmax(vnGd_x_ros_ratios)])
            gdBp_ros_max_ratios.append(gdBp_x_ros_ratios[np.argmax(gdBp_x_ros_ratios)])
            occl_ros_max_ratios.append(occl_x_ros_ratios[np.argmax(occl_x_ros_ratios)])
            sknl_ros_max_ratios.append(sknl_x_ros_ratios[np.argmax(sknl_x_ros_ratios)])
            sgrd_ros_max_ratios.append(sgrd_x_ros_ratios[np.argmax(sgrd_x_ros_ratios)])
            lime_ros_max_ratios.append(lime_x_ros_ratios[np.argmax(lime_x_ros_ratios)])
        
        # mean values
        if not ('ros' in stab_type):
            itGd_ris_mean_ratios.append(np.mean(itGd_x_ris_ratios))
            iXGd_ris_mean_ratios.append(np.mean(iXGd_x_ris_ratios))
            dLif_ris_mean_ratios.append(np.mean(dLif_x_ris_ratios))
            lwrp_ris_mean_ratios.append(np.mean(lwrp_x_ris_ratios))
            smoo_ris_mean_ratios.append(np.mean(smoo_x_ris_ratios))
            vnGd_ris_mean_ratios.append(np.mean(vnGd_x_ris_ratios))
            gdBp_ris_mean_ratios.append(np.mean(gdBp_x_ris_ratios))
            occl_ris_mean_ratios.append(np.mean(occl_x_ris_ratios))
            sknl_ris_mean_ratios.append(np.mean(sknl_x_ris_ratios))
            sgrd_ris_mean_ratios.append(np.mean(sgrd_x_ris_ratios))
            lime_ris_mean_ratios.append(np.mean(lime_x_ris_ratios))
        
        if not ('ris' in stab_type):
            itGd_ros_mean_ratios.append(np.mean(itGd_x_ros_ratios))
            iXGd_ros_mean_ratios.append(np.mean(iXGd_x_ros_ratios))
            dLif_ros_mean_ratios.append(np.mean(dLif_x_ros_ratios))
            lwrp_ros_mean_ratios.append(np.mean(lwrp_x_ros_ratios))
            smoo_ros_mean_ratios.append(np.mean(smoo_x_ros_ratios))
            vnGd_ros_mean_ratios.append(np.mean(vnGd_x_ros_ratios))
            gdBp_ros_mean_ratios.append(np.mean(gdBp_x_ros_ratios))
            occl_ros_mean_ratios.append(np.mean(occl_x_ros_ratios))
            sknl_ros_mean_ratios.append(np.mean(sknl_x_ros_ratios))
            sgrd_ros_mean_ratios.append(np.mean(sgrd_x_ros_ratios))
            lime_ros_mean_ratios.append(np.mean(lime_x_ros_ratios))
        
    
    # ------------------------------------ RETURN ratios considering all data processed
    """ OLD
    if not ('ros' in stab_type):
        itGd_ris_max= itGd_ris_max_ratios[np.argmax(itGd_ris_max_ratios)]
        iXGd_ris_max= iXGd_ris_max_ratios[np.argmax(iXGd_ris_max_ratios)]
        dLif_ris_max= dLif_ris_max_ratios[np.argmax(dLif_ris_max_ratios)]
        lwrp_ris_max= lwrp_ris_max_ratios[np.argmax(lwrp_ris_max_ratios)]
        smoo_ris_max= smoo_ris_max_ratios[np.argmax(smoo_ris_max_ratios)]
        vnGd_ris_max= vnGd_ris_max_ratios[np.argmax(vnGd_ris_max_ratios)]
        gdBp_ris_max= gdBp_ris_max_ratios[np.argmax(gdBp_ris_max_ratios)]
        occl_ris_max= occl_ris_max_ratios[np.argmax(occl_ris_max_ratios)]
        sknl_ris_max= sknl_ris_max_ratios[np.argmax(sknl_ris_max_ratios)]
        sgrd_ris_max= sgrd_ris_max_ratios[np.argmax(sgrd_ris_max_ratios)]
        lime_ris_max= lime_ris_max_ratios[np.argmax(lime_ris_max_ratios)]

    if not ('ris' in stab_type):
        itGd_ros_max= itGd_ros_max_ratios[np.argmax(itGd_ros_max_ratios)]
        iXGd_ros_max= iXGd_ros_max_ratios[np.argmax(iXGd_ros_max_ratios)]
        dLif_ros_max= dLif_ros_max_ratios[np.argmax(dLif_ros_max_ratios)]
        lwrp_ros_max= lwrp_ros_max_ratios[np.argmax(lwrp_ros_max_ratios)]
        smoo_ros_max= smoo_ros_max_ratios[np.argmax(smoo_ros_max_ratios)]
        vnGd_ros_max= vnGd_ros_max_ratios[np.argmax(vnGd_ros_max_ratios)]
        gdBp_ros_max= gdBp_ros_max_ratios[np.argmax(gdBp_ros_max_ratios)]
        occl_ros_max= occl_ros_max_ratios[np.argmax(occl_ros_max_ratios)]
        sknl_ros_max= sknl_ros_max_ratios[np.argmax(sknl_ros_max_ratios)]
        sgrd_ros_max= sgrd_ros_max_ratios[np.argmax(sgrd_ros_max_ratios)]
        lime_ros_max= lime_ros_max_ratios[np.argmax(lime_ros_max_ratios)]


    if not ('ros' in stab_type):
        itGd_ris_mean= np.mean(itGd_ris_mean_ratios)
        iXGd_ris_mean= np.mean(iXGd_ris_mean_ratios)
        dLif_ris_mean= np.mean(dLif_ris_mean_ratios)
        lwrp_ris_mean= np.mean(lwrp_ris_mean_ratios)
        smoo_ris_mean= np.mean(smoo_ris_mean_ratios)
        vnGd_ris_mean= np.mean(vnGd_ris_mean_ratios)
        gdBp_ris_mean= np.mean(gdBp_ris_mean_ratios)
        occl_ris_mean= np.mean(occl_ris_mean_ratios)
        sknl_ris_mean= np.mean(sknl_ris_mean_ratios)
        sgrd_ris_mean= np.mean(sgrd_ris_mean_ratios)
        lime_ris_mean= np.mean(lime_ris_mean_ratios)

    if not ('ris' in stab_type):
        itGd_ros_mean= np.mean(itGd_ros_mean_ratios)
        iXGd_ros_mean= np.mean(iXGd_ros_mean_ratios)
        dLif_ros_mean= np.mean(dLif_ros_mean_ratios)
        lwrp_ros_mean= np.mean(lwrp_ros_mean_ratios)
        smoo_ros_mean= np.mean(smoo_ros_mean_ratios)
        vnGd_ros_mean= np.mean(vnGd_ros_mean_ratios)
        gdBp_ros_mean= np.mean(gdBp_ros_mean_ratios)
        occl_ros_mean= np.mean(occl_ros_mean_ratios)
        sknl_ros_mean= np.mean(sknl_ros_mean_ratios)
        sgrd_ros_mean= np.mean(sgrd_ros_mean_ratios)
        lime_ros_mean= np.mean(lime_ros_mean_ratios)
    
    results= {
        'shap_kernel_ris_max': sknl_ris_max, 'shap_kernel_ris_mean': sknl_ris_mean, 
        'shap_kernel_ros_max': sknl_ros_max, 'shap_kernel_ros_mean': sknl_ros_mean, 
        
        'shap_grad_ris_max': sgrd_ris_max, 'shap_grad_ris_mean': sgrd_ris_mean, 
        'shap_grad_ros_max': sgrd_ros_max, 'shap_grad_ros_mean': sgrd_ros_mean, 
        
        'lime_ris_max': lime_ris_max, 'lime_ris_mean': lime_ris_mean, 
        'lime_ros_max': lime_ros_max, 'lime_ros_mean': lime_ros_mean, 
        
        'itGd_ris_max': itGd_ris_max, 'itGd_ris_mean': itGd_ris_mean,
        'itGd_ros_max': itGd_ros_max, 'itGd_ros_mean': itGd_ros_mean,
        
        'iXGd_ris_max': iXGd_ris_max, 'iXGd_ris_mean': iXGd_ris_mean,
        'iXGd_ros_max': iXGd_ros_max, 'iXGd_ros_mean': iXGd_ros_mean,
        
        'dLif_ris_max': dLif_ris_max, 'dLif_ris_mean': dLif_ris_mean,
        'dLif_ros_max': dLif_ros_max, 'dLif_ros_mean': dLif_ros_mean,
        
        'lwrp_ris_max': lwrp_ris_max, 'lwrp_ris_mean': lwrp_ris_mean,
        'lwrp_ros_max': lwrp_ros_max, 'lwrp_ros_mean': lwrp_ros_mean,
        
        'smoothG_ris_max': smoo_ris_max, 'smoothG_ris_mean': smoo_ris_mean,
        'smoothG_ros_max': smoo_ros_max, 'smoothG_ros_mean': smoo_ros_mean,
        
        'vanillaG_ris_max': vnGd_ris_max, 'vanillaG_ris_mean': vnGd_ris_mean,
        'vanillaG_ros_max': vnGd_ros_max, 'vanillaG_ros_mean': vnGd_ros_mean,
        
        'GuidBprop_ris_max': gdBp_ris_max, 'GuidBprop_ris_mean': gdBp_ris_mean,
        'GuidBprop_ros_max': gdBp_ros_max, 'GuidBprop_ros_mean': gdBp_ros_mean,
        
        'occlusion_ris_max': occl_ris_max, 'occlusion_ris_mean': occl_ris_mean,
        'occlusion_ros_max': occl_ros_max, 'occlusion_ros_mean': occl_ros_mean,   
    }
    """
    
    exps= descriptor['methods']
    
    if not ('ros' in stab_type):
        if ('IntegratedGradients' in exps):
            itGd_ris= pd.DataFrame(data= np.asarray(np.stack((itGd_ris_max_ratios,itGd_ris_mean_ratios))).T, 
                                   columns=['itGd_ris_max','itGd_ris_mean'])
        if ('InputXGradient' in exps): 
            iXGd_ris= pd.DataFrame(data= np.asarray(np.stack((iXGd_ris_max_ratios,iXGd_ris_mean_ratios))).T, 
                                   columns=['iXGd_ris_max','iXGd_ris_mean'])
        if ('DeepLift' in exps): 
            dLif_ris= pd.DataFrame(data= np.asarray(np.stack((dLif_ris_max_ratios,dLif_ris_mean_ratios))).T, 
                                   columns=['dLif_ris_max','dLif_ris_mean'])
        if ('LRP' in exps): 
            lwrp_ris= pd.DataFrame(data= np.asarray(np.stack((lwrp_ris_max_ratios,lwrp_ris_mean_ratios))).T, 
                                   columns=['lwrp_ris_max','lwrp_ris_mean'])
        if ('SmoothGrad' in exps): 
            smoo_ris= pd.DataFrame(data= np.asarray(np.stack((smoo_ris_max_ratios,smoo_ris_mean_ratios))).T, 
                                   columns=['smoo_ris_max','smoo_ris_mean'])
        if ('VanillaGrad' in exps):
            vnGd_ris= pd.DataFrame(data= np.asarray(np.stack((vnGd_ris_max_ratios,vnGd_ris_mean_ratios))).T, 
                                   columns=['vnGd_ris_max','vnGd_ris_mean'])
        if ('GuidedBackprop' in exps): 
            gdBp_ris= pd.DataFrame(data= np.asarray(np.stack((gdBp_ris_max_ratios,gdBp_ris_mean_ratios))).T, 
                                   columns=['gdBp_ris_max','gdBp_ris_mean'])
        if ('Occlusion' in exps): 
            occl_ris= pd.DataFrame(data= np.asarray(np.stack((occl_ris_max_ratios,occl_ris_mean_ratios))).T, 
                                   columns=['occl_ris_max','occl_ris_mean'])
        if ('Lime' in exps): 
            lime_ris= pd.DataFrame(data= np.asarray(np.stack((lime_ris_max_ratios,lime_ris_mean_ratios))).T, 
                                   columns=['lime_ris_max','lime_ris_mean'])
        if ('KernelShap' in exps): 
            sknl_ris= pd.DataFrame(data= np.asarray(np.stack((sknl_ris_max_ratios,sknl_ris_mean_ratios))).T, 
                                   columns=['knlSh_ris_max','knlSh_ris_mean'])
        if ('GradientShap' in exps): 
            sgrd_ris= pd.DataFrame(data= np.asarray(np.stack((sgrd_ris_max_ratios,sgrd_ris_mean_ratios))).T, 
                                   columns=['grdSh_ris_max','grdSh_ris_mean'])
        
    if not ('ris' in stab_type):
        if ('IntegratedGradients' in exps):
            itGd_ros= pd.DataFrame(data= np.asarray(np.stack((itGd_ros_max_ratios,itGd_ros_mean_ratios))).T, 
                                   columns=['itGd_ros_max','itGd_ros_mean'])
        if ('InputXGradient' in exps): 
            iXGd_ros= pd.DataFrame(data= np.asarray(np.stack((iXGd_ros_max_ratios,iXGd_ros_mean_ratios))).T, 
                                   columns=['iXGd_ros_max','iXGd_ros_mean'])
        if ('DeepLift' in exps): 
            dLif_ros= pd.DataFrame(data= np.asarray(np.stack((dLif_ros_max_ratios,dLif_ros_mean_ratios))).T, 
                                   columns=['dLif_ros_max','dLif_ros_mean'])
        if ('LRP' in exps): 
            lwrp_ros= pd.DataFrame(data= np.asarray(np.stack((lwrp_ros_max_ratios,lwrp_ros_mean_ratios))).T, 
                                   columns=['lwrp_ros_max','lwrp_ros_mean'])
        if ('SmoothGrad' in exps): 
            smoo_ros= pd.DataFrame(data= np.asarray(np.stack((smoo_ros_max_ratios,smoo_ros_mean_ratios))).T, 
                                   columns=['smoo_ros_max','smoo_ros_mean'])
        if ('VanillaGrad' in exps):
            vnGd_ros= pd.DataFrame(data= np.asarray(np.stack((vnGd_ros_max_ratios,vnGd_ros_mean_ratios))).T, 
                                   columns=['vnGd_ros_max','vnGd_ros_mean'])
        if ('GuidedBackprop' in exps): 
            gdBp_ros= pd.DataFrame(data= np.asarray(np.stack((gdBp_ros_max_ratios,gdBp_ros_mean_ratios))).T, 
                                   columns=['gdBp_ros_max','gdBp_ros_mean'])
        if ('Occlusion' in exps): 
            occl_ros= pd.DataFrame(data= np.asarray(np.stack((occl_ros_max_ratios,occl_ros_mean_ratios))).T, 
                                   columns=['occl_ros_max','occl_ros_mean'])
        if ('Lime' in exps): 
            lime_ros= pd.DataFrame(data= np.asarray(np.stack((lime_ros_max_ratios,lime_ros_mean_ratios))).T, 
                                   columns=['lime_ros_max','lime_ros_mean'])
        if ('KernelShap' in exps): 
            sknl_ros= pd.DataFrame(data= np.asarray(np.stack((sknl_ros_max_ratios,sknl_ros_mean_ratios))).T, 
                                   columns=['knlSh_ros_max','knlSh_ros_mean'])
        if ('GradientShap' in exps): 
            sgrd_ros= pd.DataFrame(data= np.asarray(np.stack((sgrd_ros_max_ratios,sgrd_ros_mean_ratios))).T, 
                                   columns=['grdSh_ros_max','grdSh_ros_mean'])
            
    results= data.reset_index(drop=True)
    
    if ('IntegratedGradients' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,itGd_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,itGd_ros], axis=1)
    if ('InputXGradient' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,iXGd_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,iXGd_ros], axis=1)
    if ('DeepLift' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,dLif_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,dLif_ros], axis=1)
    if ('LRP' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,lwrp_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,lwrp_ros], axis=1)
    if ('SmoothGrad' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,smoo_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,smoo_ros], axis=1)
    if ('VanillaGrad' in exps):
        if not ('ros' in stab_type): results= pd.concat([results,vnGd_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,vnGd_ros], axis=1)
    if ('GuidedBackprop' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,gdBp_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,gdBp_ros], axis=1)
    if ('Occlusion' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,occl_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,occl_ros], axis=1)
    if ('Lime' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,lime_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,lime_ros], axis=1)
    if ('KernelShap' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,sknl_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,sknl_ros], axis=1)
    if ('GradientShap' in exps): 
        if not ('ros' in stab_type): results= pd.concat([results,sgrd_ris], axis=1)
        if not ('ris' in stab_type): results= pd.concat([results,sgrd_ros], axis=1)
    
    # the max/mean stability ratios
    return results

In [17]:
# defining the perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.01
perturbation_flip_percentage= 0.0001

perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

In [36]:
# defining a parameters' descriptor dictionary

descriptor= dict()

# settings used for the metrics
descriptor['num_samples']= 20
descriptor['num_perts']= 10
descriptor['pert_max_distance']= 0.01
descriptor['num_runs']= 10
descriptor['feature_metadata']= ['c'] * (train_ox.shape[1])   # c means continuous features
descriptor['p_norm']= 2
descriptor['eps_norm']= 1e-6
descriptor['eps_eval_add']= 0.05
descriptor['methods']= ['IntegratedGradients', 'InputXGradient', 'KernelShap']

# full list of options for 'methods': 
#    ['IntegratedGradients','InputXGradient','DeepLift','LRP','SmoothGrad',
#     'VanillaGrad','GuidedBackprop','Occlusion','Lime','KernelShap','GradientShap']

In [419]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
# DataFrame version -- values per input instance and XAI method

descriptor['methods']= ['IntegratedGradients','KernelShap']

print('RIS/ROS --') 
relative_stability(nn_pytorch_model_ox, train_ox[0:2], labels_train_ox[0:2], 
                   perturbation, descriptor, stab_type='both')

RIS/ROS --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0



,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,...,ft_19,ft_20,itGd_ris_max,itGd_ris_mean,itGd_ros_max,itGd_ros_mean,knlSh_ris_max,knlSh_ris_mean,knlSh_ros_max,knlSh_ros_mean
0,0.705423,0.250877,0.393413,0.353529,0.328286,0.374804,0.312217,0.130608,0.281738,0.368790,...,0.382225,0.347964,851.075534,357.175341,20039.396347,3350.452487,272.062740,128.459907,2966.323243,888.984807
1,0.119699,0.665472,0.235255,0.404961,0.292727,0.321720,0.246230,0.324422,0.360535,0.228708,...,0.448307,0.455760,11.081008,6.563031,1.688872,0.579519,124.281357,68.916080,33.613824,6.713440


In [196]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
# OLD version listing aggregated values
print('RIS/ROS --') 
relative_stability(nn_pytorch_model_ox, train_ox[0:2], labels_train_ox[0:2], 
                   perturbation, descriptor)

RIS/ROS --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0



{'shap_kernel_ris_max': 288.41167262576096,
 'shap_kernel_ris_mean': 128.0006990072635,
 'shap_kernel_ros_max': 3208.7747183198667,
 'shap_kernel_ros_mean': 430.9475043281932,
 'shap_grad_ris_max': 2976.4110283118007,
 'shap_grad_ris_mean': 407.6447993182081,
 'shap_grad_ros_max': 14649.287381405265,
 'shap_grad_ros_mean': 2125.999526906572,
 'lime_ris_max': 603.2448828791422,
 'lime_ris_mean': 128.81309902355082,
 'lime_ros_max': 4396.26521366086,
 'lime_ros_mean': 617.6114392773858,
 'itGd_ris_max': 1264.1202840334388,
 'itGd_ris_mean': 186.2550745808265,
 'itGd_ros_max': 5211.688543955314,
 'itGd_ros_mean': 768.6886989470687,
 'iXGd_ris_max': 148.21659287316552,
 'iXGd_ris_mean': 15.703968422167241,
 'iXGd_ros_max': 4076.7007855026827,
 'iXGd_ros_mean': 230.18868318697136,
 'dLif_ris_max': 148.21659287316442,
 'dLif_ris_mean': 15.703968422167149,
 'dLif_ros_max': 4076.700785502652,
 'dLif_ros_mean': 230.18868318696968,
 'lwrp_ris_max': 148.2169012897293,
 'lwrp_ris_mean': 15.7039857

# Metric -- Run Explanation Stability -- RES

In [18]:
# Run Explanation Stability
# model is a trained PyTorch classifier
# data is a preprocessed Pandas DataFrame to evaluate
# labels are the respective data labels (Pandas DataFrame)
# descriptor defines the parameters to explanations and data perturbations

# RETURN a measure of stability from multiple runs over non-perturbed x.
# the greater the value, the less stable the method is. Methods:
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        LRP
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion
#        LIME
#        KernelSHAP
#        GradientSHAP

# The larger the RES values of the underlying explanation method, the more unstable the method 
# to reiterations.

def run_stability(model, data, labels, descriptor):
    
    itGd_stability_ratios= []
    iXGd_stability_ratios= []
    dLif_stability_ratios= []
    lwrp_stability_ratios= []
    smoo_stability_ratios= []
    vnGd_stability_ratios= []
    gdBp_stability_ratios= []
    occl_stability_ratios= []
    sknl_stability_ratios= []
    sgrd_stability_ratios= []
    lime_stability_ratios= []
    
    
    runs= int(descriptor['num_runs'])
    data_size= data.shape[0]
    
    for i_data in pb.progressbar(range(data_size), "Progress: ", 40):
        
        # i_data and its label as pd.DataFrames
        target_x= pd.DataFrame(data=[data.iloc[i_data,:]], columns=data.columns)
        target_y= pd.DataFrame(data=[labels.iloc[i_data]], columns=labels.columns)
        
        # i_data and its label as tensors
        x_data= torch.tensor(np.asarray(target_x), dtype=torch.float64)
        y_data= torch.tensor(np.asarray(target_y), dtype=torch.float64)
        
        
        # data point prediction
        fx_data= model(x_data)
        y_pred = log_odds_to_binary_class(fx_data)

        
        itGd_x_exps= []
        iXGd_x_exps= []
        dLif_x_exps= []
        lwrp_x_exps= []
        smoo_x_exps= []
        vnGd_x_exps= []
        gdBp_x_exps= []
        occl_x_exps= []
        sknl_x_exps= []
        sgrd_x_exps= []
        lime_x_exps= []
        
               
        for i in range(runs):
            # ------------------------------------ get x_data explanation -- n runs
            x_exps= get_x_explanations(model, x_data, y_data, descriptor)
        
            itGd_x_exp= x_exps['itGd']
            iXGd_x_exp= x_exps['iXGd']
            dLif_x_exp= x_exps['dLif']
            lwrp_x_exp= x_exps['lwrp']
            smoo_x_exp= x_exps['smoo']
            vnGd_x_exp= x_exps['vnGd']
            gdBp_x_exp= x_exps['gdBp']
            occl_x_exp= x_exps['occl']
            lime_x_exp= x_exps['lime']
            sknl_x_exp= x_exps['k_shap']
            sgrd_x_exp= x_exps['g_shap']
            
            
            # get the explanation of each method to each run
            itGd_x_exps.append(itGd_x_exp.numpy())
            iXGd_x_exps.append(iXGd_x_exp.numpy())
            dLif_x_exps.append(dLif_x_exp.numpy())
            lwrp_x_exps.append(lwrp_x_exp.numpy())
            smoo_x_exps.append(smoo_x_exp.numpy())
            vnGd_x_exps.append(vnGd_x_exp.numpy())
            gdBp_x_exps.append(gdBp_x_exp.numpy())
            occl_x_exps.append(occl_x_exp.numpy())
            sknl_x_exps.append(sknl_x_exp.numpy())
            sgrd_x_exps.append(sgrd_x_exp.numpy())
            lime_x_exps.append(lime_x_exp.numpy())
            
            
        itGd_x_exps= np.asarray(itGd_x_exps)
        iXGd_x_exps= np.asarray(iXGd_x_exps)
        dLif_x_exps= np.asarray(dLif_x_exps)
        lwrp_x_exps= np.asarray(lwrp_x_exps)
        smoo_x_exps= np.asarray(smoo_x_exps)
        vnGd_x_exps= np.asarray(vnGd_x_exps)
        gdBp_x_exps= np.asarray(gdBp_x_exps)
        occl_x_exps= np.asarray(occl_x_exps)
        sknl_x_exps= np.asarray(sknl_x_exps)
        sgrd_x_exps= np.asarray(sgrd_x_exps)
        lime_x_exps= np.asarray(lime_x_exps)
        
        
        itGd_x_exps_mean= np.mean(itGd_x_exps, axis=0)
        iXGd_x_exps_mean= np.mean(iXGd_x_exps, axis=0)
        dLif_x_exps_mean= np.mean(dLif_x_exps, axis=0)
        lwrp_x_exps_mean= np.mean(lwrp_x_exps, axis=0)
        smoo_x_exps_mean= np.mean(smoo_x_exps, axis=0)
        vnGd_x_exps_mean= np.mean(vnGd_x_exps, axis=0)
        gdBp_x_exps_mean= np.mean(gdBp_x_exps, axis=0)
        occl_x_exps_mean= np.mean(occl_x_exps, axis=0)
        sknl_x_exps_mean= np.mean(sknl_x_exps, axis=0)
        sgrd_x_exps_mean= np.mean(sgrd_x_exps, axis=0)
        lime_x_exps_mean= np.mean(lime_x_exps, axis=0)
        
        
        itGd_x_exp_ratios= []
        iXGd_x_exp_ratios= []
        dLif_x_exp_ratios= []
        lwrp_x_exp_ratios= []
        smoo_x_exp_ratios= []
        vnGd_x_exp_ratios= []
        gdBp_x_exp_ratios= []
        occl_x_exp_ratios= []
        sknl_x_exp_ratios= []
        sgrd_x_exp_ratios= []
        lime_x_exp_ratios= []
        
        
        # ------------------------------------ distance of each explanation from the mean of explanations 
        for j in range(runs):
            itGd_x_exp_ratios.append(lp_norm_dif(itGd_x_exps_mean, itGd_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))

            iXGd_x_exp_ratios.append(lp_norm_dif(iXGd_x_exps_mean, iXGd_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))

            dLif_x_exp_ratios.append(lp_norm_dif(dLif_x_exps_mean, dLif_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))

            lwrp_x_exp_ratios.append(lp_norm_dif(lwrp_x_exps_mean, lwrp_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))

            smoo_x_exp_ratios.append(lp_norm_dif(smoo_x_exps_mean, smoo_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))

            vnGd_x_exp_ratios.append(lp_norm_dif(vnGd_x_exps_mean, vnGd_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))

            gdBp_x_exp_ratios.append(lp_norm_dif(gdBp_x_exps_mean, gdBp_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))

            occl_x_exp_ratios.append(lp_norm_dif(occl_x_exps_mean, occl_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))
            
            sknl_x_exp_ratios.append(lp_norm_dif(sknl_x_exps_mean, sknl_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))
            
            sgrd_x_exp_ratios.append(lp_norm_dif(sgrd_x_exps_mean, sgrd_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))

            lime_x_exp_ratios.append(lp_norm_dif(lime_x_exps_mean, lime_x_exps[j], 
                                          p_norm=descriptor['p_norm'], norm=False))
        
            
        # ------------------------------------ max ratio related to each x_data
        itGd_stability_ratios.append(itGd_x_exp_ratios[np.argmax(itGd_x_exp_ratios)])
        iXGd_stability_ratios.append(iXGd_x_exp_ratios[np.argmax(iXGd_x_exp_ratios)])
        dLif_stability_ratios.append(dLif_x_exp_ratios[np.argmax(dLif_x_exp_ratios)])
        lwrp_stability_ratios.append(lwrp_x_exp_ratios[np.argmax(lwrp_x_exp_ratios)])
        smoo_stability_ratios.append(smoo_x_exp_ratios[np.argmax(smoo_x_exp_ratios)])
        vnGd_stability_ratios.append(vnGd_x_exp_ratios[np.argmax(vnGd_x_exp_ratios)])
        gdBp_stability_ratios.append(gdBp_x_exp_ratios[np.argmax(gdBp_x_exp_ratios)])
        occl_stability_ratios.append(occl_x_exp_ratios[np.argmax(occl_x_exp_ratios)])
        sknl_stability_ratios.append(sknl_x_exp_ratios[np.argmax(sknl_x_exp_ratios)])
        sgrd_stability_ratios.append(sgrd_x_exp_ratios[np.argmax(sgrd_x_exp_ratios)])
        lime_stability_ratios.append(lime_x_exp_ratios[np.argmax(lime_x_exp_ratios)])
        
    
    # ------------------------------------ general max ratio related to all data
    """ OLD
    itGd_max= itGd_stability_ratios[np.argmax(itGd_stability_ratios)]
    iXGd_max= iXGd_stability_ratios[np.argmax(iXGd_stability_ratios)]
    dLif_max= dLif_stability_ratios[np.argmax(dLif_stability_ratios)]
    lwrp_max= lwrp_stability_ratios[np.argmax(lwrp_stability_ratios)]
    smoo_max= smoo_stability_ratios[np.argmax(smoo_stability_ratios)]
    vnGd_max= vnGd_stability_ratios[np.argmax(vnGd_stability_ratios)]
    gdBp_max= gdBp_stability_ratios[np.argmax(gdBp_stability_ratios)]
    occl_max= occl_stability_ratios[np.argmax(occl_stability_ratios)]
    sknl_max= sknl_stability_ratios[np.argmax(sknl_stability_ratios)]
    sgrd_max= sgrd_stability_ratios[np.argmax(sgrd_stability_ratios)]
    lime_max= lime_stability_ratios[np.argmax(lime_stability_ratios)]
    
    results= {
        'itGd_res': itGd_max,
        'iXGd_res': iXGd_max,
        'dLif_res': dLif_max,
        'lwrp_res': lwrp_max,
        'smoothG_res': smoo_max,
        'vanillaG_res': vnGd_max,
        'GuidBprop_res': gdBp_max,
        'occlusion_res': occl_max,
        'shap_kernel_res': sknl_max,
        'shap_grad_res': sgrd_max,
        'lime_res': lime_max,
    }
    """
    
    exps= descriptor['methods']
    
    results= data.reset_index(drop=True)
    
    if ('IntegratedGradients' in exps): 
        itGd_stability_ratios= pd.DataFrame(data= np.asarray(itGd_stability_ratios).T, 
                                            columns=['itGd_max_res'])
        results= pd.concat([results,itGd_stability_ratios], axis=1)
    if ('InputXGradient' in exps): 
        iXGd_stability_ratios= pd.DataFrame(data= np.asarray(iXGd_stability_ratios).T, 
                                            columns=['iXGd_max_res'])
        results= pd.concat([results,iXGd_stability_ratios], axis=1)
    if ('DeepLift' in exps): 
        dLif_stability_ratios= pd.DataFrame(data= np.asarray(dLif_stability_ratios).T, 
                                            columns=['dLif_max_res'])
        results= pd.concat([results,dLif_stability_ratios], axis=1)
    if ('LRP' in exps): 
        lwrp_stability_ratios= pd.DataFrame(data= np.asarray(lwrp_stability_ratios).T, 
                                            columns=['lwrp_max_res'])
        results= pd.concat([results,lwrp_stability_ratios], axis=1)
    if ('SmoothGrad' in exps): 
        smoo_stability_ratios= pd.DataFrame(data= np.asarray(smoo_stability_ratios).T, 
                                            columns=['smoo_max_res'])
        results= pd.concat([results,smoo_stability_ratios], axis=1)
    if ('VanillaGrad' in exps):
        vnGd_stability_ratios= pd.DataFrame(data= np.asarray(vnGd_stability_ratios).T, 
                                            columns=['vnGd_max_res'])
        results= pd.concat([results,vnGd_stability_ratios], axis=1)
    if ('GuidedBackprop' in exps): 
        gdBp_stability_ratios= pd.DataFrame(data= np.asarray(gdBp_stability_ratios).T, 
                                            columns=['gdBp_max_res'])
        results= pd.concat([results,gdBp_stability_ratios], axis=1)
    if ('Occlusion' in exps): 
        occl_stability_ratios= pd.DataFrame(data= np.asarray(occl_stability_ratios).T, 
                                            columns=['occl_max_res'])
        results= pd.concat([results,occl_stability_ratios], axis=1)
    if ('Lime' in exps): 
        lime_stability_ratios= pd.DataFrame(data= np.asarray(lime_stability_ratios).T, 
                                            columns=['lime_max_res'])
        results= pd.concat([results,lime_stability_ratios], axis=1)
    if ('KernelShap' in exps): 
        sknl_stability_ratios= pd.DataFrame(data= np.asarray(sknl_stability_ratios).T, 
                                            columns=['knlSh_max_res'])
        results= pd.concat([results,sknl_stability_ratios], axis=1)
    if ('GradientShap' in exps): 
        sgrd_stability_ratios= pd.DataFrame(data= np.asarray(sgrd_stability_ratios).T, 
                                            columns=['grdSh_max_res'])
        results= pd.concat([results,sgrd_stability_ratios], axis=1)
    
    
    # max stability_ratios
    return results

In [435]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
# DataFrame version -- values per input instance and XAI method

descriptor['methods']= ['IntegratedGradients','KernelShap']

print('RES --')
run_stability(nn_pytorch_model_ox, train_ox[0:2], labels_train_ox[0:2], descriptor)

RES --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0



,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,...,ft_13,ft_14,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20,itGd_max_res,knlSh_max_res
0,0.705423,0.250877,0.393413,0.353529,0.328286,0.374804,0.312217,0.130608,0.281738,0.368790,...,0.440911,0.48193,0.263193,0.348485,0.127619,0.420548,0.382225,0.347964,1.117830e-15,2.789396
1,0.119699,0.665472,0.235255,0.404961,0.292727,0.321720,0.246230,0.324422,0.360535,0.228708,...,0.328190,0.44591,0.344165,0.403461,0.579152,0.558943,0.448307,0.455760,2.136503e-15,2.896336


In [40]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
# OLD version listing aggregated values
print('RES --')
run_stability(nn_pytorch_model_ox, train_ox[0:2], labels_train_ox[0:2], descriptor)

RES --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0


{'itGd_res': 2.136503349211554e-15,
 'iXGd_res': 2.3241876662647787e-15,
 'dLif_res': 2.3188782643185826e-15,
 'lwrp_res': 2.767218361258043e-15,
 'smoothG_res': 0.9428568769372766,
 'vanillaG_res': 5.064613052477648e-15,
 'GuidBprop_res': 5.064613052477648e-15,
 'occlusion_res': 9.396006e-07,
 'shap_kernel_res': 3.018952,
 'shap_grad_res': 4.119925837922512,
 'lime_res': 2.411296}

# Metric -- Prediction Gap on Important Features -- PGI

In [19]:
# convert log odds values to predicted probability

def log_odds_to_pred_proba(lodds):
    
    odds= np.exp(lodds)
    
    p= odds / (1 + odds)
    
    return p

In [20]:
# RETURN the index of an ascending sorted array if reverse is False, descending sorted if reverse is True

def sorted_indices(seq, reverse:bool=False):
    seq= np.asarray(seq)
    
    if reverse: seq= -seq
    
    return [i for (v, i) in sorted((v, i) for (i, v) in enumerate(seq))]

In [21]:
# x is a Pandas DataFrame with an instance
# e_index is a vector of indices from an explanation ordered with sorted_indices()
# top_k is an INTEGER representing the number of top features
# x_pert represents the zero/perturbed instance from x used to generate a x'

# RETURN a perturbed instance xi' [i not in top_k]

def get_top_k_x_noise(x, e_index, top_k, x_pert):
    
    # get the top k features
    top_k_index= e_index[:top_k]
    names= x.columns

    top_k_names= [names[v] for (i, v) in enumerate(top_k_index)]
    non_top_k_names= list(filter(lambda x:x not in top_k_names, names))
    
    x_pert_sample= pd.DataFrame(data=[x_pert.numpy()], columns=x.columns)

    # delete/perturb features that are not in the top k ones to produce x'
    x_noise= (x.copy()).reset_index(drop=True)
    x_noise[non_top_k_names]= x_pert_sample[non_top_k_names]

    return x_noise

In [22]:
# Prediction Gap on Important Features
# model is a trained PyTorch classifier
# data is a preprocessed Pandas DataFrame to assess
# labels are the data labels (Pandas DataFrame)
# descriptor defines the parameters to explanations and data perturbations
# top_k an integer or a list of integers with the number of top k features to assess each XAI method
# noise_type represents the type of perturbation: 'zero' places zeros to non-important features (deletion),
#    or 'pert' uses OpenXAI perturbation with parameter m given by descriptor['num_perts']
# perturbation is a OpenXAI perturbation object required when noise_type is not 'zero'

# RETURN PGI metric considering all features as important for 
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        LRP
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion
#        LIME
#        KernelSHAP
#        GradientSHAP

# PGI measures the predictive faithfulness based on the change in a model’s prediction probability from
# the selection of k features deemed as the most important ones and determined by a post-hoc XAI method.
# The higher the PGI value, the more faithful the explanation.

# see Dai, Jessica, et al. "Fairness via explanation quality: Evaluating disparities in the quality of 
# post hoc explanations," Agarwal, Chirag, et al. "OpenXAI: Towards a transparent evaluation of 
# model explanations," and Ortigossa, Evandro S., Thales Gonçalves, and Luis Gustavo Nonato. "EXplainable 
# Artificial Intelligence (XAI)-From Theory to Methods and Applications."

def eval_pred_faithfulness(model, data, labels, descriptor, top_k= 1, noise_type='zero', perturbation=None):
    
    top_k= list(np.reshape([top_k], -1))
    top_k= [ int(x) for x in top_k ]     # ensure everything is integer
    
    for i in range(len(top_k)):          # ensure upper and lower boundaries
        if (top_k[i]< 0): top_k[i]= 0
        elif (top_k[i]> data.columns.shape[0]): top_k[i]= data.columns.shape[0]
        
    top_k= list(np.unique(top_k))        # remove repetitions
    

    itGd_pgi= [[] for _ in range(len(top_k))]
    iXGd_pgi= [[] for _ in range(len(top_k))]
    dLif_pgi= [[] for _ in range(len(top_k))]
    lwrp_pgi= [[] for _ in range(len(top_k))]
    smoo_pgi= [[] for _ in range(len(top_k))]
    vnGd_pgi= [[] for _ in range(len(top_k))]
    gdBp_pgi= [[] for _ in range(len(top_k))]
    occl_pgi= [[] for _ in range(len(top_k))]
    sknl_pgi= [[] for _ in range(len(top_k))]
    sgrd_pgi= [[] for _ in range(len(top_k))]
    lime_pgi= [[] for _ in range(len(top_k))]
        
        
    data_size= data.shape[0]
    
    m_size= descriptor['num_perts']
    if ('zero' in noise_type): m_size= 1
        
    
    for i_data in pb.progressbar(range(data_size), "Progress: ", 40):
        
        # i_data and its label as pd.DataFrames
        target_x= pd.DataFrame(data=[data.iloc[i_data,:]], columns=data.columns)
        target_y= pd.DataFrame(data=[labels.iloc[i_data]], columns=labels.columns)
        
        # i_data and its label as tensors
        x_data= torch.tensor(np.asarray(target_x), dtype=torch.float64)
        y_data= torch.tensor(np.asarray(target_y), dtype=torch.float64)
        
        
        # get the predicted probability of f(x)
        fx_acc= log_odds_to_pred_proba(model(x_data).detach().numpy())
        #fx_cls= log_odds_to_binary_class(model(x_data))
        
        
        # ------------------------------------ get x_data explanation
        x_exps= get_x_explanations(model, x_data, y_data, descriptor)
        
        itGd_x_exp= x_exps['itGd']
        iXGd_x_exp= x_exps['iXGd']
        dLif_x_exp= x_exps['dLif']
        lwrp_x_exp= x_exps['lwrp']
        smoo_x_exp= x_exps['smoo']
        vnGd_x_exp= x_exps['vnGd']
        gdBp_x_exp= x_exps['gdBp']
        occl_x_exp= x_exps['occl']
        lime_x_exp= x_exps['lime']
        sknl_x_exp= x_exps['k_shap']
        sgrd_x_exp= x_exps['g_shap']
        
        
        # ------------------------------------ get the sorted indices from most important features
        itGd_importances= (itGd_x_exp.numpy())
        itGd_importances_index= sorted_indices(itGd_importances, reverse=True)

        iXGd_importances= (iXGd_x_exp.numpy())
        iXGd_importances_index= sorted_indices(iXGd_importances, reverse=True)

        dLif_importances= (dLif_x_exp.numpy())
        dLif_importances_index= sorted_indices(dLif_importances, reverse=True)

        lwrp_importances= (lwrp_x_exp.numpy())
        lwrp_importances_index= sorted_indices(lwrp_importances, reverse=True)

        smoo_importances= (smoo_x_exp.numpy())
        smoo_importances_index= sorted_indices(smoo_importances, reverse=True)

        vnGd_importances= (vnGd_x_exp.numpy())
        vnGd_importances_index= sorted_indices(vnGd_importances, reverse=True)

        gdBp_importances= (gdBp_x_exp.numpy())
        gdBp_importances_index= sorted_indices(gdBp_importances, reverse=True)

        occl_importances= (occl_x_exp.numpy())
        occl_importances_index= sorted_indices(occl_importances, reverse=True)
        
        sknl_importances= (sknl_x_exp.numpy())
        sknl_importances_index= sorted_indices(sknl_importances, reverse=True)
        
        sgrd_importances= (sgrd_x_exp.numpy())
        sgrd_importances_index= sorted_indices(sgrd_importances, reverse=True)
            
        lime_importances= (lime_x_exp.numpy())
        lime_importances_index= sorted_indices(lime_importances, reverse=True)
        
        
        if not ('zero' in noise_type):
            # ------------------------------------ x_data perturbation case. generate a sample
            # data point perturbation
            mask= torch.zeros(x_data.reshape(-1).shape, dtype=torch.bool)

            x_pert= perturbation.get_perturbed_inputs(original_sample=x_data.reshape(-1),
                                                      feature_mask=mask,
                                                      num_samples=m_size,
                                                      max_distance=descriptor['pert_max_distance'],
                                                      feature_metadata=descriptor['feature_metadata'])
        else: x_pert= torch.zeros(x_data.shape)
            
        
        # ------------------------------------ iterate for each top_k value of each explanation
        for j in range(len(top_k)):
            top_k_value= top_k[j]
            
            itGd_fx_acc_diff= []
            iXGd_fx_acc_diff= []
            dLif_fx_acc_diff= []
            lwrp_fx_acc_diff= []
            smoo_fx_acc_diff= []
            vnGd_fx_acc_diff= []
            gdBp_fx_acc_diff= []
            occl_fx_acc_diff= []
            sknl_fx_acc_diff= []
            sgrd_fx_acc_diff= []
            lime_fx_acc_diff= []
            
            for m in range(m_size):
            
                # delete/perturb features that are not in the top k ones to produce x'
                itGd_target_x= get_top_k_x_noise(target_x, itGd_importances_index, top_k_value, x_pert[m])
                iXGd_target_x= get_top_k_x_noise(target_x, iXGd_importances_index, top_k_value, x_pert[m])
                dLif_target_x= get_top_k_x_noise(target_x, dLif_importances_index, top_k_value, x_pert[m])
                lwrp_target_x= get_top_k_x_noise(target_x, lwrp_importances_index, top_k_value, x_pert[m])
                smoo_target_x= get_top_k_x_noise(target_x, smoo_importances_index, top_k_value, x_pert[m])
                vnGd_target_x= get_top_k_x_noise(target_x, vnGd_importances_index, top_k_value, x_pert[m])
                gdBp_target_x= get_top_k_x_noise(target_x, gdBp_importances_index, top_k_value, x_pert[m])
                occl_target_x= get_top_k_x_noise(target_x, occl_importances_index, top_k_value, x_pert[m])
                sknl_target_x= get_top_k_x_noise(target_x, sknl_importances_index, top_k_value, x_pert[m])
                sgrd_target_x= get_top_k_x_noise(target_x, sgrd_importances_index, top_k_value, x_pert[m])
                lime_target_x= get_top_k_x_noise(target_x, lime_importances_index, top_k_value, x_pert[m])
            
                # get f(x')
                itGd_fx_acc= log_odds_to_pred_proba(model(torch.tensor(itGd_target_x.values)).detach().numpy())
                iXGd_fx_acc= log_odds_to_pred_proba(model(torch.tensor(iXGd_target_x.values)).detach().numpy())
                dLif_fx_acc= log_odds_to_pred_proba(model(torch.tensor(dLif_target_x.values)).detach().numpy())
                lwrp_fx_acc= log_odds_to_pred_proba(model(torch.tensor(lwrp_target_x.values)).detach().numpy())
                smoo_fx_acc= log_odds_to_pred_proba(model(torch.tensor(smoo_target_x.values)).detach().numpy())
                vnGd_fx_acc= log_odds_to_pred_proba(model(torch.tensor(vnGd_target_x.values)).detach().numpy())
                gdBp_fx_acc= log_odds_to_pred_proba(model(torch.tensor(gdBp_target_x.values)).detach().numpy())
                occl_fx_acc= log_odds_to_pred_proba(model(torch.tensor(occl_target_x.values)).detach().numpy())
                sknl_fx_acc= log_odds_to_pred_proba(model(torch.tensor(sknl_target_x.values)).detach().numpy())
                sgrd_fx_acc= log_odds_to_pred_proba(model(torch.tensor(sgrd_target_x.values)).detach().numpy())
                lime_fx_acc= log_odds_to_pred_proba(model(torch.tensor(lime_target_x.values)).detach().numpy())
            
                # take the difference between f(x) and each m f(x')
                itGd_fx_acc_diff.append(np.abs(fx_acc - itGd_fx_acc).flatten()[0])
                iXGd_fx_acc_diff.append(np.abs(fx_acc - iXGd_fx_acc).flatten()[0])
                dLif_fx_acc_diff.append(np.abs(fx_acc - dLif_fx_acc).flatten()[0])
                lwrp_fx_acc_diff.append(np.abs(fx_acc - lwrp_fx_acc).flatten()[0])
                smoo_fx_acc_diff.append(np.abs(fx_acc - smoo_fx_acc).flatten()[0])
                vnGd_fx_acc_diff.append(np.abs(fx_acc - vnGd_fx_acc).flatten()[0])
                gdBp_fx_acc_diff.append(np.abs(fx_acc - gdBp_fx_acc).flatten()[0])
                occl_fx_acc_diff.append(np.abs(fx_acc - occl_fx_acc).flatten()[0])
                sknl_fx_acc_diff.append(np.abs(fx_acc - sknl_fx_acc).flatten()[0])
                sgrd_fx_acc_diff.append(np.abs(fx_acc - sgrd_fx_acc).flatten()[0])
                lime_fx_acc_diff.append(np.abs(fx_acc - lime_fx_acc).flatten()[0])
                
            # compute the mean (1/m)sum(|f(x) - f(x'm)|)
            # high fidelity explanation will result in low differences, so we inverted the value to get 
            # the higher the PGI value, the more faithful the explanation
            itGd_pgi[j].append(1- np.mean(itGd_fx_acc_diff))
            iXGd_pgi[j].append(1- np.mean(iXGd_fx_acc_diff))
            dLif_pgi[j].append(1- np.mean(dLif_fx_acc_diff))
            lwrp_pgi[j].append(1- np.mean(lwrp_fx_acc_diff))
            smoo_pgi[j].append(1- np.mean(smoo_fx_acc_diff))
            vnGd_pgi[j].append(1- np.mean(vnGd_fx_acc_diff))
            gdBp_pgi[j].append(1- np.mean(gdBp_fx_acc_diff))
            occl_pgi[j].append(1- np.mean(occl_fx_acc_diff))
            sknl_pgi[j].append(1- np.mean(sknl_fx_acc_diff))
            sgrd_pgi[j].append(1- np.mean(sgrd_fx_acc_diff))
            lime_pgi[j].append(1- np.mean(lime_fx_acc_diff))
                 
    """
    itGd_m_pgi= (np.mean(itGd_pgi, axis=1))
    #itGd_m_pgi_sd= np.std(itGd_pgi, axis=1)
    iXGd_m_pgi= (np.mean(iXGd_pgi, axis=1))
    #iXGd_m_pgi_sd= np.std(iXGd_pgi, axis=1)
    dLif_m_pgi= (np.mean(dLif_pgi, axis=1))
    #dLif_m_pgi_sd= np.std(dLif_pgi, axis=1)
    lwrp_m_pgi= (np.mean(lwrp_pgi, axis=1))
    #lwrp_m_pgi_sd= np.std(lwrp_pgi, axis=1)
    smoo_m_pgi= (np.mean(smoo_pgi, axis=1))
    #smoo_m_pgi_sd= np.std(smoo_pgi, axis=1)
    vnGd_m_pgi= (np.mean(vnGd_pgi, axis=1))
    #vnGd_m_pgi_sd= np.std(vnGd_pgi, axis=1)
    gdBp_m_pgi= (np.mean(gdBp_pgi, axis=1))
    #gdBp_m_pgi_sd= np.std(gdBp_pgi, axis=1)
    occl_m_pgi= (np.mean(occl_pgi, axis=1))
    #occl_m_pgi_sd= np.std(occl_pgi, axis=1)
    sknl_m_pgi= (np.mean(sknl_pgi, axis=1))
    #sknl_m_pgi_sd= np.std(sknl_pgi, axis=1)
    sgrd_m_pgi= (np.mean(sgrd_pgi, axis=1))
    #sgrd_m_pgi_sd= np.std(sgrd_pgi, axis=1)
    lime_m_pgi= (np.mean(lime_pgi, axis=1))
    #lime_m_pgi_sd= np.std(lime_pgi, axis=1)
    
    results= {
        'PGI values from top k features': top_k,
        'itGd pgi': list(itGd_m_pgi),
        'iXGd pgi': list(iXGd_m_pgi),
        'dLif pgi': list(dLif_m_pgi),
        'lwrp pgi': list(lwrp_m_pgi),
        'smoothG pgi': list(smoo_m_pgi),
        'vanillaG pgi': list(vnGd_m_pgi),
        'GuidBprop pgi': list(gdBp_m_pgi),
        'occlusion pgi': list(occl_m_pgi),
        'shap_kernel pgi': list(sknl_m_pgi),
        'shap_grad pgi': list(sgrd_m_pgi),
        'lime pgi': list(lime_m_pgi),
    }
    """
    
    exps= descriptor['methods']
    
    results= data.reset_index(drop=True)
    
    def df_top_k_col_labels(method_name, k_vect):
        return [method_name + '_PGI_top_' + str(v) for (i, v) in enumerate(k_vect)]

    
    if ('IntegratedGradients' in exps): 
        itGd_pgi_ratios= pd.DataFrame(data= np.asarray(itGd_pgi).T, 
                                            columns=df_top_k_col_labels('itGd', top_k))
        results= pd.concat([results,itGd_pgi_ratios], axis=1)
    if ('InputXGradient' in exps): 
        iXGd_pgi_ratios= pd.DataFrame(data= np.asarray(iXGd_pgi).T, 
                                            columns=df_top_k_col_labels('iXGd', top_k))
        results= pd.concat([results,iXGd_pgi_ratios], axis=1)
    if ('DeepLift' in exps): 
        dLif_pgi_ratios= pd.DataFrame(data= np.asarray(dLif_pgi).T, 
                                            columns=df_top_k_col_labels('dLif', top_k))
        results= pd.concat([results,dLif_pgi_ratios], axis=1)
    if ('LRP' in exps): 
        lwrp_pgi_ratios= pd.DataFrame(data= np.asarray(lwrp_pgi).T, 
                                            columns=df_top_k_col_labels('lwrp', top_k))
        results= pd.concat([results,lwrp_pgi_ratios], axis=1)
    if ('SmoothGrad' in exps): 
        smoo_pgi_ratios= pd.DataFrame(data= np.asarray(smoo_pgi).T, 
                                            columns=df_top_k_col_labels('smoo', top_k))
        results= pd.concat([results,smoo_pgi_ratios], axis=1)
    if ('VanillaGrad' in exps):
        vnGd_pgi_ratios= pd.DataFrame(data= np.asarray(vnGd_pgi).T, 
                                            columns=df_top_k_col_labels('vnGd', top_k))
        results= pd.concat([results,vnGd_pgi_ratios], axis=1)
    if ('GuidedBackprop' in exps): 
        gdBp_pgi_ratios= pd.DataFrame(data= np.asarray(gdBp_pgi).T, 
                                            columns=df_top_k_col_labels('gdBp', top_k))
        results= pd.concat([results,gdBp_pgi_ratios], axis=1)
    if ('Occlusion' in exps): 
        occl_pgi_ratios= pd.DataFrame(data= np.asarray(occl_pgi).T, 
                                            columns=df_top_k_col_labels('occl', top_k))
        results= pd.concat([results,occl_pgi_ratios], axis=1)
    if ('Lime' in exps): 
        lime_pgi_ratios= pd.DataFrame(data= np.asarray(lime_pgi).T, 
                                            columns=df_top_k_col_labels('lime', top_k))
        results= pd.concat([results,lime_pgi_ratios], axis=1)
    if ('KernelShap' in exps): 
        sknl_pgi_ratios= pd.DataFrame(data= np.asarray(sknl_pgi).T, 
                                            columns=df_top_k_col_labels('knlSh', top_k))
        results= pd.concat([results,sknl_pgi_ratios], axis=1)
    if ('GradientShap' in exps): 
        sgrd_pgi_ratios= pd.DataFrame(data= np.asarray(sgrd_pgi).T, 
                                            columns=df_top_k_col_labels('grdSh', top_k))
        results= pd.concat([results,sgrd_pgi_ratios], axis=1)
    
    return results

In [721]:
# evaluate explanation faithfulness deleting non important features -- using 20 instances from train_ox
# 'pert' -- perturbing non-important top-k

descriptor['num_perts']= 20
descriptor['methods']= ['IntegratedGradients','KernelShap']

print('PGI --')
eval_pred_faithfulness(nn_pytorch_model_ox, train_ox[0:20], labels_train_ox[0:20], 
                       descriptor, top_k= [3,5], noise_type='pert', perturbation=perturbation)

PGI --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0



,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,...,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20,itGd_PGI_top_3,itGd_PGI_top_5,knlSh_PGI_top_3,knlSh_PGI_top_5
0,0.705423,0.250877,0.393413,0.353529,0.328286,0.374804,0.312217,0.130608,0.281738,0.368790,...,0.263193,0.348485,0.127619,0.420548,0.382225,0.347964,0.990841,0.990842,0.989850,0.989114
1,0.119699,0.665472,0.235255,0.404961,0.292727,0.321720,0.246230,0.324422,0.360535,0.228708,...,0.344165,0.403461,0.579152,0.558943,0.448307,0.455760,0.954182,0.959474,0.954182,0.955716
2,0.226558,0.670225,0.344595,0.497451,0.298231,0.316184,0.430817,0.313977,0.414127,0.390934,...,0.609983,0.332080,0.327387,0.239984,0.463658,0.595457,0.999919,0.999918,0.999928,0.999926
3,0.229718,0.291506,0.394624,0.390758,0.310606,0.680242,0.283559,0.345412,0.421726,0.394265,...,0.437791,0.589921,0.648241,0.700112,0.269150,0.385109,0.999718,0.999714,0.999718,0.999714
4,0.373117,0.249822,0.259671,0.329655,0.307589,0.657956,0.408162,0.318289,0.333588,0.428981,...,0.562405,0.525431,0.416705,0.713325,0.586582,0.639019,0.980956,0.980637,0.980572,0.980637
5,0.395901,0.295885,0.863933,0.313462,0.268225,0.361538,0.323815,0.376219,0.451750,0.262446,...,0.488711,0.597548,0.374620,0.485374,0.495549,0.576942,0.999827,0.999833,0.999827,0.999833
6,0.229636,0.343321,0.259064,0.351755,0.271810,0.292448,0.237793,0.407361,0.246277,0.792101,...,0.382245,0.432302,0.395263,0.303847,0.517135,0.294609,0.970944,0.976923,0.970944,0.979252
7,0.486406,0.338815,0.281680,0.437508,0.314528,0.524643,0.280526,0.340006,0.391516,0.292202,...,0.595534,0.425392,0.354826,0.591445,0.137266,0.364208,0.978425,0.978414,0.976375,0.981584
8,0.222800,0.289054,0.719015,0.268941,0.270206,0.338865,0.176528,0.230221,0.285404,0.296298,...,0.608251,0.438549,0.639854,0.468186,0.555627,0.299158,0.999144,0.999167,0.999183,0.999236
9,0.257216,0.365027,0.332741,0.303198,0.728844,0.377062,0.143351,0.433963,0.252325,0.315574,...,0.495169,0.350798,0.286425,0.568375,0.376313,0.602393,0.997663,0.997706,0.997663,0.997706


In [693]:
# evaluate explanation faithfulness deleting non important features -- using 20 instances from train_ox
# 'pert' -- perturbing non-important top-k

descriptor['num_perts']= 20
descriptor['methods']= ['IntegratedGradients','InputXGradient','DeepLift','LRP','SmoothGrad','VanillaGrad','Occlusion','Lime','KernelShap','GradientShap']

print('PGI --')
eval_pred_faithfulness(nn_pytorch_model_ox, train_ox[0:20], labels_train_ox[0:20], 
                       descriptor, top_k= [3,5], noise_type='pert', perturbation)

PGI --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0



{'PGI values from top k features': [3, 5],
 'itGd pgi': [0.9843048070182112, 0.9868304395475125],
 'iXGd pgi': [0.9853824368349574, 0.9873260980832799],
 'dLif pgi': [0.9853824368349574, 0.9873260980832799],
 'lwrp pgi': [0.9853824368349574, 0.9873260980832799],
 'smoothG pgi': [0.9843091008467605, 0.9868505252063103],
 'vanillaG pgi': [0.9855558970322319, 0.9872772384259909],
 'GuidBprop pgi': [0.9808868864193421, 0.9848405562488759],
 'occlusion pgi': [0.9853709607857324, 0.9865769161309516],
 'shap_kernel pgi': [0.9840507188830626, 0.986679821159893],
 'shap_grad pgi': [0.9844714265566343, 0.9863818901492147],
 'lime pgi': [0.9842843822809704, 0.9858752039314433]}

In [717]:
# evaluate explanation faithfulness deleting non important features -- using 20 instances from train_ox
# 'zero' -- deleting non-important top-k

descriptor['methods']= ['IntegratedGradients','KernelShap']

print('PGI --')
eval_pred_faithfulness(nn_pytorch_model_ox, train_ox[0:20], labels_train_ox[0:20], 
                       descriptor, top_k= [3,5], noise_type='zero')

PGI --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0



,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,...,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20,itGd_PGI_top_3,itGd_PGI_top_5,knlSh_PGI_top_3,knlSh_PGI_top_5
0,0.705423,0.250877,0.393413,0.353529,0.328286,0.374804,0.312217,0.130608,0.281738,0.368790,...,0.263193,0.348485,0.127619,0.420548,0.382225,0.347964,0.060808,0.060370,0.060889,0.060660
1,0.119699,0.665472,0.235255,0.404961,0.292727,0.321720,0.246230,0.324422,0.360535,0.228708,...,0.344165,0.403461,0.579152,0.558943,0.448307,0.455760,0.532677,0.531766,0.532169,0.531766
2,0.226558,0.670225,0.344595,0.497451,0.298231,0.316184,0.430817,0.313977,0.414127,0.390934,...,0.609983,0.332080,0.327387,0.239984,0.463658,0.595457,0.999435,0.999010,0.999344,0.998954
3,0.229718,0.291506,0.394624,0.390758,0.310606,0.680242,0.283559,0.345412,0.421726,0.394265,...,0.437791,0.589921,0.648241,0.700112,0.269150,0.385109,0.003161,0.003354,0.003161,0.003411
4,0.373117,0.249822,0.259671,0.329655,0.307589,0.657956,0.408162,0.318289,0.333588,0.428981,...,0.562405,0.525431,0.416705,0.713325,0.586582,0.639019,0.114000,0.114089,0.117592,0.114089
5,0.395901,0.295885,0.863933,0.313462,0.268225,0.361538,0.323815,0.376219,0.451750,0.262446,...,0.488711,0.597548,0.374620,0.485374,0.495549,0.576942,0.002743,0.002655,0.003098,0.002655
6,0.229636,0.343321,0.259064,0.351755,0.271810,0.292448,0.237793,0.407361,0.246277,0.792101,...,0.382245,0.432302,0.395263,0.303847,0.517135,0.294609,0.337611,0.337757,0.337611,0.337542
7,0.486406,0.338815,0.281680,0.437508,0.314528,0.524643,0.280526,0.340006,0.391516,0.292202,...,0.595534,0.425392,0.354826,0.591445,0.137266,0.364208,0.124929,0.125176,0.124929,0.124881
8,0.222800,0.289054,0.719015,0.268941,0.270206,0.338865,0.176528,0.230221,0.285404,0.296298,...,0.608251,0.438549,0.639854,0.468186,0.555627,0.299158,0.007563,0.008027,0.007874,0.007364
9,0.257216,0.365027,0.332741,0.303198,0.728844,0.377062,0.143351,0.433963,0.252325,0.315574,...,0.495169,0.350798,0.286425,0.568375,0.376313,0.602393,0.013699,0.013957,0.013962,0.013707


In [695]:
# evaluate explanation faithfulness deleting non important features -- using 20 instances from train_ox
# 'zero' -- deleting non-important top-k

descriptor['methods']= ['IntegratedGradients','InputXGradient','DeepLift','LRP','SmoothGrad','VanillaGrad','Occlusion','Lime','KernelShap','GradientShap']

print('PGI --')
eval_pred_faithfulness(nn_pytorch_model_ox, train_ox[0:20], labels_train_ox[0:20], 
                       descriptor, top_k= [3,5], noise_type='zero')

PGI --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0



{'PGI values from top k features': [3, 5],
 'itGd pgi': [0.28169537993715277, 0.2815596144786611],
 'iXGd pgi': [0.4176388948778338, 0.35396488544340354],
 'dLif pgi': [0.4176388948778338, 0.35396488544340354],
 'lwrp pgi': [0.4176388948778338, 0.35396488544340354],
 'smoothG pgi': [0.28169537993715277, 0.28158863177019783],
 'vanillaG pgi': [0.41276016295366097, 0.33496336029983553],
 'GuidBprop pgi': [0.7479705644916359, 0.5606920103569367],
 'occlusion pgi': [0.362876350831871, 0.32939115310811673],
 'shap_kernel pgi': [0.28224814682963817, 0.2817007595517532],
 'shap_grad pgi': [0.2817535385958082, 0.28159179254548217],
 'lime pgi': [0.28166911493506275, 0.28148777060210217]}

# Metric -- Local Accuracy Preservation -- LAP

In [23]:
# convert predicted probability log values

def pred_proba_to_log(p):
    
    l= np.log(p)
    
    return l

In [24]:
# convert log predicted probability values

def log_to_pred_proba(l):
    
    p= np.exp(l)
    
    return p

In [25]:
# convert probabilities to log odds
# eps is a small positive value to prevent division by zero

def pred_proba_to_log_odds(p, eps=1e-9):
    
    p_clipped= np.clip(p, eps, (1 - eps))
    
    return np.log(p_clipped / (1 - p_clipped))

In [26]:
from sklearn.utils import shuffle

# get a random sample from train data and labels to reduce the size

def data_sample(data, labels, sample_size):
    
    sh_data= shuffle(data)
    sh_data= sh_data[:sample_size]
    
    sh_data_index= sh_data.index
    
    sh_labels= labels.iloc[sh_data_index]
    
    sh_data= sh_data.reset_index(drop=True)
    sh_labels= sh_labels.reset_index(drop=True)
    
    return sh_data, sh_labels

In [27]:
# return a DataFrame with replaced values (mean/median/mode/zeros/none to categorical and numeric) 
# to fill train cols. df is post-processed (cat_cols encoded)

def replace_values(df,num_cols,num_type='mean',cat_type='none'):
    cat_values= None
    num_values= None
    
    if (cat_type=='mean'):
        cat_values= df.mean(axis=0).to_frame().T
    elif (cat_type=='median'):
        cat_values= df.median(axis=0).to_frame().T
    elif (cat_type=='mode'):
        cat_values= df.mode(axis=0)
    elif (cat_type=='zeros'):
        cat_values= df.loc[0:0,:].copy()
        cat_values.loc[:,:]= 0
    
    if (num_type=='mode'):
        num_values= df.mode(axis=0)
    elif (num_type=='median'):
        num_values= df.median(axis=0).to_frame().T
    elif (num_type=='mean'):
        num_values= df.mean(axis=0).to_frame().T
    elif (num_type=='zeros'):
        num_values= df.loc[0:0,:].copy()
        num_values.loc[:,:]= 0

    if(cat_type!='none'):
        cat_values[num_cols]= num_values[num_cols]
        return cat_values
    
    return num_values

In [28]:
# Expected value can be understood as the average model output across the training set, and the true labels
# SHAP paper mention a dataset "would be predicted if we did not know any features"

# The 'absence of features' or better 'not knowing the feature' needs to be defined/considered carefully. 
# In the context of SHAP it doesn't meant that Xi=0 but it means that we do not know the value of Xi
# but we still may know the distribution of potential values of Xi or we could estimate this distribution 
# based on the data, in practice, a mean value is considered, and then, we average over this predicted 
# probs to each label.

# Receive the a dataset sample (from train) and take for every item in our sample, 
# the average of the predicted probabilities

# RETURN the baseline value phi_0 related to the average prediction of our classifier

def expected_value_x_mean(model, x_data):
    
    # Expected value over the mean from x_data
    tam= x_data.size()
    
    prob_preds= model(x_data)
        
    avg_pred= torch.mean(prob_preds)
    
    return avg_pred

In [29]:
# Local Accuracy Preservation
# model is a trained PyTorch classifier
# data is a preprocessed Pandas DataFrame to assess
# labels are the data labels (Pandas DataFrame)
# descriptor defines the parameters to explanations and data perturbations
# train_data is the Pandas DataFrame with model's training data
# labels_train is the Pandas DataFrame with model's training labels data

# RETURN LAP metric considering all features as important for 
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        LRP
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion
#        LIME
#        KernelSHAP
#        GradientSHAP

# For an additive explanator, local accuracy preservation means that the sum of all feature importance values
# will be equal to the difference between the expected value of the model and the predicted value
# That is, f(x) = phi_0 + Sum(phi_i), with phi_0 = E[f(X)].

# RETURN a measure of faithfulness for local accuracy preservation for each explainer over a non-perturbed 
#        dataset as a ratio of explanations that preserved local accuracy (mean value, the greater the value,
#        the less faithful the method is)

def eval_local_accuracy(model, data, labels, train_data, labels_train, descriptor):
    
    data_size= data.shape[0]
    
    # sample when data is too large
    sample_size= 10000
    if (data_size> sample_size): 
        sample_train_data, sample_labels_train= data_sample(train_data, labels_train, sample_size)
    else:
        sample_train_data= train_data
        sample_labels_train= labels_train
    
    # we assume all data are numerical
    num_fts= train_data.columns
    
    #mean_inst= replace_values(sample_train_data, num_fts, num_type='mean', cat_type='median')
    
    # e_fx can be understood as the average model output across the training set X when Xi is not known
    phi_0_logit= expected_value_x_mean(model, torch.from_numpy(sample_train_data.astype(float).values))
    phi_0_logit= phi_0_logit.detach().numpy()
    phi_0= log_odds_to_pred_proba(phi_0_logit)
    
    
    itGd_lap= 0
    iXGd_lap= 0
    dLif_lap= 0
    lwrp_lap= 0
    smoo_lap= 0
    vnGd_lap= 0
    gdBp_lap= 0
    occl_lap= 0
    sknl_lap= 0
    sgrd_lap= 0
    lime_lap= 0
    
    
    tolerance= descriptor['eps_eval_add']
    
    for i_data in pb.progressbar(range(data_size), "Progress: ", 40):
        
        # i_data and its label as pd.DataFrames
        target_x= pd.DataFrame(data=[data.iloc[i_data,:]], columns=data.columns)
        target_y= pd.DataFrame(data=[labels.iloc[i_data]], columns=labels.columns)
        
        # i_data and its label as tensors
        x_data= torch.tensor(np.asarray(target_x), dtype=torch.float64)
        y_data= torch.tensor(np.asarray(target_y), dtype=torch.float64)
        
        # get the predicted probability of f(x)
        fx_p_logit= model(x_data)
        fx_p_prob= log_odds_to_pred_proba(fx_p_logit.detach().numpy()).flatten()[0]
        pred_label= log_odds_to_binary_class(fx_p_logit).numpy()
        fx_p_logit= (fx_p_logit.detach().numpy()).flatten()[0]
        
        fx_tol_pls= fx_p_prob + tolerance
        fx_tol_min= fx_p_prob - tolerance
        
        logit_fx_tol_pls= pred_proba_to_log_odds(fx_tol_pls)
        logit_fx_tol_min= pred_proba_to_log_odds(fx_tol_min)
        
        
        # ------------------------------------ get x_data explanation
        x_exps= get_x_explanations(model, x_data, y_data, descriptor)
        
        itGd_x_exp= x_exps['itGd']
        iXGd_x_exp= x_exps['iXGd']
        dLif_x_exp= x_exps['dLif']
        lwrp_x_exp= x_exps['lwrp']
        smoo_x_exp= x_exps['smoo']
        vnGd_x_exp= x_exps['vnGd']
        gdBp_x_exp= x_exps['gdBp']
        occl_x_exp= x_exps['occl']
        lime_x_exp= x_exps['lime']
        sknl_x_exp= x_exps['k_shap']
        sgrd_x_exp= x_exps['g_shap']
        
        
        # ------------------------------------ get the additive approximation
        itGd_fx= phi_0_logit + np.sum(itGd_x_exp.numpy())
        iXGd_fx= phi_0_logit + np.sum(iXGd_x_exp.numpy())
        dLif_fx= phi_0_logit + np.sum(dLif_x_exp.numpy())
        lwrp_fx= phi_0_logit + np.sum(lwrp_x_exp.numpy())
        smoo_fx= phi_0_logit + np.sum(smoo_x_exp.numpy())
        vnGd_fx= phi_0_logit + np.sum(vnGd_x_exp.numpy())
        gdBp_fx= phi_0_logit + np.sum(gdBp_x_exp.numpy())
        occl_fx= phi_0_logit + np.sum(occl_x_exp.numpy())
        sknl_fx= phi_0_logit + np.sum(sknl_x_exp.numpy())
        sgrd_fx= phi_0_logit + np.sum(sgrd_x_exp.numpy())
        lime_fx= phi_0_logit + np.sum(lime_x_exp.numpy())
        
        
        if (itGd_fx>= logit_fx_tol_min and itGd_fx<= logit_fx_tol_pls):
            itGd_lap= itGd_lap + 1

        if (iXGd_fx>= logit_fx_tol_min and iXGd_fx<= logit_fx_tol_pls):
            iXGd_lap= iXGd_lap + 1

        if (dLif_fx>= logit_fx_tol_min and dLif_fx<= logit_fx_tol_pls):
            dLif_lap= dLif_lap + 1

        if (lwrp_fx>= logit_fx_tol_min and lwrp_fx<= logit_fx_tol_pls):
            lwrp_lap= lwrp_lap + 1

        if (smoo_fx>= logit_fx_tol_min and smoo_fx<= logit_fx_tol_pls):
            smoo_lap= smoo_lap + 1

        if (vnGd_fx>= logit_fx_tol_min and vnGd_fx<= logit_fx_tol_pls):
            vnGd_lap= vnGd_lap + 1

        if (gdBp_fx>= logit_fx_tol_min and gdBp_fx<= logit_fx_tol_pls):
            gdBp_lap= gdBp_lap + 1

        if (occl_fx>= logit_fx_tol_min and occl_fx<= logit_fx_tol_pls):
            occl_lap= occl_lap + 1
                
        if (sknl_fx>= logit_fx_tol_min and sknl_fx<= logit_fx_tol_pls):
            sknl_lap= sknl_lap + 1
        
        if (sgrd_fx>= logit_fx_tol_min and sgrd_fx<= logit_fx_tol_pls):
            sgrd_lap= sgrd_lap + 1
        
        if (lime_fx>= logit_fx_tol_min and lime_fx<= logit_fx_tol_pls):
            lime_lap= lime_lap + 1
        
        
    itGd_lap= itGd_lap / data_size
    iXGd_lap= iXGd_lap / data_size
    dLif_lap= dLif_lap / data_size
    lwrp_lap= lwrp_lap / data_size
    smoo_lap= smoo_lap / data_size
    vnGd_lap= vnGd_lap / data_size
    gdBp_lap= gdBp_lap / data_size
    occl_lap= occl_lap / data_size
    sknl_lap= sknl_lap / data_size
    sgrd_lap= sgrd_lap / data_size
    lime_lap= lime_lap / data_size
    
    """
    results= {
        'itGd LAP': itGd_lap,
        'iXGd LAP': iXGd_lap,
        'dLif LAP': dLif_lap,
        'lwrp LAP': lwrp_lap,
        'smoo LAP': smoo_lap,
        'vnGd LAP': vnGd_lap,
        'gdBp LAP': gdBp_lap,
        'occl LAP': occl_lap,
        'shap_kernel LAP': sknl_lap,
        'shap_grad LAP': sgrd_lap,
        'lime LAP': lime_lap,
    }
    """
    
    exps= descriptor['methods']
    
    results=[]
    
    if ('IntegratedGradients' in exps): 
        itGd_lap= ['IntegratedGradients', itGd_lap]
        results.append(itGd_lap)
    if ('InputXGradient' in exps): 
        iXGd_lap= ['InputXGradient', iXGd_lap]
        results.append(iXGd_lap)
    if ('DeepLift' in exps): 
        dLif_lap= ['DeepLift', dLif_lap]
        results.append(dLif_lap)
    if ('LRP' in exps): 
        lwrp_lap= ['LRP', lwrp_lap]
        results.append(lwrp_lap)
    if ('SmoothGrad' in exps): 
        smoo_lap= ['SmoothGrad', smoo_lap]
        results.append(smoo_lap)
    if ('VanillaGrad' in exps):
        vnGd_lap= ['VanillaGrad', vnGd_lap]
        results.append(vnGd_lap)
    if ('GuidedBackprop' in exps): 
        gdBp_lap= ['GuidedBackprop', gdBp_lap]
        results.append(gdBp_lap)
    if ('Occlusion' in exps): 
        occl_lap= ['Occlusion', occl_lap]
        results.append(occl_lap)
    if ('Lime' in exps): 
        lime_lap= ['Lime', lime_lap]
        results.append(lime_lap)
    if ('KernelShap' in exps): 
        sknl_lap= ['KernelShap', sknl_lap]
        results.append(sknl_lap)
    if ('GradientShap' in exps): 
        sgrd_lap= ['GradientShap', sgrd_lap]
        results.append(sgrd_lap)
    
    return pd.DataFrame(data=results, columns=['Explainer','LAP'])

In [865]:
# evaluate local accuracy preservation -- using instances from test_ox
descriptor_ox['eps_eval_add']= 0.1

descriptor['methods']= ['IntegratedGradients','InputXGradient','DeepLift','LRP','SmoothGrad','VanillaGrad','Occlusion','Lime','KernelShap','GradientShap']

print('LAP --')
eval_local_accuracy(nn_pytorch_model_ox, test_ox, labels_test_ox, 
                    train_ox, labels_train_ox, descriptor)

LAP --
Progress: [████████████████████████████████████████] 100.0% - Est wait 00:0.0



,Explainer,LAP
0,IntegratedGradients,0.380
1,InputXGradient,0.270
2,DeepLift,0.270
3,LRP,0.270
4,SmoothGrad,0.365
5,VanillaGrad,0.175
6,Occlusion,0.375
7,Lime,0.375
8,KernelShap,0.380
9,GradientShap,0.400


# Test settings

In [30]:
ox_path= 'data/synth_OX_20/processed/'

train_ox= pd.read_csv(ox_path + 'X_train.csv')
test_ox = pd.read_csv(ox_path + 'X_test.csv')
labels_train_ox= pd.read_csv(ox_path + 'y_train.csv')
labels_test_ox = pd.read_csv(ox_path + 'y_test.csv')

train_ox.head()

,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,ft_11,ft_12,ft_13,ft_14,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20
0,0.705423,0.250877,0.393413,0.353529,0.328286,0.374804,0.312217,0.130608,0.281738,0.368790,0.758551,0.608015,0.440911,0.481930,0.263193,0.348485,0.127619,0.420548,0.382225,0.347964
1,0.119699,0.665472,0.235255,0.404961,0.292727,0.321720,0.246230,0.324422,0.360535,0.228708,0.630680,0.377279,0.328190,0.445910,0.344165,0.403461,0.579152,0.558943,0.448307,0.455760
2,0.226558,0.670225,0.344595,0.497451,0.298231,0.316184,0.430817,0.313977,0.414127,0.390934,0.637601,0.566323,0.420372,0.513464,0.609983,0.332080,0.327387,0.239984,0.463658,0.595457
3,0.229718,0.291506,0.394624,0.390758,0.310606,0.680242,0.283559,0.345412,0.421726,0.394265,0.589889,0.494661,0.640523,0.478399,0.437791,0.589921,0.648241,0.700112,0.269150,0.385109
4,0.373117,0.249822,0.259671,0.329655,0.307589,0.657956,0.408162,0.318289,0.333588,0.428981,0.631567,0.439209,0.392151,0.282901,0.562405,0.525431,0.416705,0.713325,0.586582,0.639019


In [31]:
import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.neural_network import MLPClassifier

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(activation='relu', alpha=0.0001, hidden_layer_sizes=(64, 64, 64), 
                            learning_rate_init=0.01, max_iter=500, random_state=0, solver='sgd')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.835

In [32]:
# Create a custom PyTorch model that mimics the behavior of a scikit-learn MLPClassifier model

import torch.nn as nn

# Define the PyTorch Neural Network model
class MLPClassifierModel(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(MLPClassifierModel, self).__init__()
        
        self.layers= nn.ModuleList([nn.Linear(input_size, hidden_sizes[0])])
        self.activations= [nn.ReLU()]
        
        for i in range(1, len(hidden_sizes)):
            self.layers.append(nn.Linear(hidden_sizes[i-1], hidden_sizes[i]))
            self.activations.append(nn.ReLU())
        
        self.output_layer= nn.Linear(hidden_sizes[-1], output_size)

        
    def forward(self, x):
        for layer, activation in zip(self.layers, self.activations):
            x= activation(layer(x))
        
        x= self.output_layer(x)
        
        return x

In [33]:
# convert a scikit-learn NN model to a PyTorch NN model

# skl_nn_model is the scikit-learn Neural Net model
# input_size is the number of input features

# RETURN a PyTorch Neural Net model used to binary classifications

def sklearn_to_pytorch_NN(skl_nn_model, input_size):

    # Convert the scikit-learn model to a PyTorch model
    input_size= input_size
    hidden_sizes= skl_nn_model.hidden_layer_sizes
    output_size= 1  # binary classification -- one output neuron for the binary prediction

    nn_pytorch_model= MLPClassifierModel(input_size, hidden_sizes, output_size)

    # Transfer the weights from the scikit-learn model to the PyTorch model
    for i, layer in enumerate(nn_pytorch_model.layers):
        layer.weight.data= torch.tensor(skl_nn_model.coefs_[i].T, dtype=torch.float64)
        layer.bias.data= torch.tensor(skl_nn_model.intercepts_[i], dtype=torch.float64)

    nn_pytorch_model.output_layer.weight.data= torch.tensor(skl_nn_model.coefs_[-1].T, dtype=torch.float64)
    nn_pytorch_model.output_layer.bias.data= torch.tensor(skl_nn_model.intercepts_[-1], dtype=torch.float64)
    
    
    return nn_pytorch_model

In [34]:
nn_pytorch_model_ox= sklearn_to_pytorch_NN(nn3_model_ox, train_ox.shape[1])
target_i= pd.DataFrame(data=[train_ox.iloc[0,:]], columns=train_ox.columns)
target_l= pd.DataFrame(data=[labels_train_ox.iloc[0]], columns=labels_train_ox.columns)
x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float64)

In [35]:
pred= nn_pytorch_model_ox(x_data_tensor)

pred= log_odds_to_binary_class(pred)

pred

tensor([[0.]], dtype=torch.float64)

In [23]:
exp= explainer.Explainers()

# SHAP Explainer from SHAP

target_i= pd.DataFrame(data=[train_ox.iloc[0,:]], columns=train_ox.columns)

shap_values= exp.shap(nn3_model_ox, train_ox, target_i)
print("shap_values =", shap_values)

shap_values = tensor([-0.1487, -0.0362, -0.0746, -0.1233,  0.0079,  0.0108,  0.0004, -0.1046,
        -0.0292, -0.0500,  0.1092, -0.0025, -0.0042,  0.0054, -0.0679,  0.0021,
         0.1275,  0.0067, -0.1054, -0.0233], dtype=torch.float64)


In [24]:
# KernelSHAP Explainer from SHAP

shap_values= exp.k_shap(nn3_model_ox, train_ox, target_i)
print("shap_values =", shap_values)

shap_values = tensor([-0.1564, -0.0559, -0.0734, -0.1056,  0.0090,  0.0088,  0.0000, -0.0985,
        -0.0315, -0.0479,  0.1014,  0.0000, -0.0036,  0.0049, -0.0600,  0.0000,
         0.1216,  0.0063, -0.1037, -0.0156], dtype=torch.float64)


In [25]:
# KernelSHAP from captum

nn_pytorch_model_ox= sklearn_to_pytorch_NN(nn3_model_ox, train_ox.shape[1])

target_i= pd.DataFrame(data=[train_ox.iloc[0,:]], columns=train_ox.columns)
x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float32)

# Explain predictions on test data
attributions= exp.c_kshap(nn_pytorch_model_ox, x_data_tensor)

# Print attributions for a sample data point
print("SHAP Attributions:", attributions)

SHAP Attributions: tensor([ 8.3634e-01, -4.2812e-01, -2.8547e+00,  3.0064e+00, -1.1360e+00,
         5.4667e-01,  3.0849e+00,  1.9103e-01, -9.0052e-01, -6.6046e+00,
        -1.9769e-01, -1.4805e+00,  5.5948e-01, -5.9485e-01,  4.8116e-01,
         6.1416e-03, -1.3453e+00,  6.1207e-01,  1.2902e+00,  8.5864e-02])


In [50]:
descriptor['feature_metadata']= ['c'] * (train_ox.shape[1])   # c means continuous features

descriptor['num_runs']= 20
descriptor['methods']= ['IntegratedGradients','InputXGradient','DeepLift','LRP','SmoothGrad','GuidedBackprop','VanillaGrad','Occlusion','Lime','KernelShap','GradientShap']

print('RES --')
run_stability(nn_pytorch_model_ox, test_ox, labels_test_ox, descriptor)

RES --
Progress: [████████████████████████████████████████] 100.0% - Elapsed time 11:59 min   



{'itGd_res': 6.5648830852894715e-15,
 'iXGd_res': 8.479952941588347e-15,
 'dLif_res': 8.042222936750132e-15,
 'lwrp_res': 9.602346779957892e-15,
 'smoothG_res': 2.2320636214634986,
 'vanillaG_res': 1.5872695158824184e-14,
 'GuidBprop_res': 1.5872695158824184e-14,
 'occlusion_res': 3.0762892e-06,
 'shap_kernel_res': 6.657044,
 'shap_grad_res': 9.941520764945546,
 'lime_res': 3.9857814}

In [40]:
indep_path= 'data/independent/processed/'

train_indep= pd.read_csv(indep_path + 'X_train.csv')
test_indep = pd.read_csv(indep_path + 'X_test.csv')
labels_train_indep= pd.read_csv(indep_path + 'y_train.csv')
labels_test_indep = pd.read_csv(indep_path + 'y_test.csv')

In [44]:
# Create the 3-hyden layers Neural Net classifier model
nn3_model_indep= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=64,
                            hidden_layer_sizes=(32, 32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn3_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn3_model_indep.predict(test_indep))
acc_nn3_indep

1.0

In [46]:
nn_pytorch_model_indep= sklearn_to_pytorch_NN(nn3_model_indep, train_indep.shape[1])

In [52]:
descriptor['feature_metadata']= ['c'] * (train_indep.shape[1])   # c means continuous features

print('RES --')
run_stability(nn_pytorch_model_indep, test_indep, labels_test_indep, descriptor)

RES --
Progress: [████████████████████████████████████████] 100.0% - Elapsed time 03:17 min   



{'itGd_res': 3.877848521283954e-15,
 'iXGd_res': 5.546672453247935e-15,
 'dLif_res': 5.479600186598906e-15,
 'lwrp_res': 5.1229905854055344e-15,
 'smoothG_res': 0.6748101327057857,
 'vanillaG_res': 1.0395861869767154e-14,
 'GuidBprop_res': 1.0395861869767154e-14,
 'occlusion_res': 2.5345994e-06,
 'shap_kernel_res': 1.9974564,
 'shap_grad_res': 3.4302815249473753,
 'lime_res': 0.9374313}